**Author**:
- Tianci Wang - [tiancwang@ethz.ch](tianci:tiancwang@ethz.ch)

**Date**: 06/08/2026

# Policy Portfolio Construction Code v1.0
This code is used to perform policy evaluation and build policy portfolios for each energy transition pathway based on the baseline transition pathway of EP2050+.
The basic assumption is that the current in-force policies are sufficient for the transition following the baseline scenario in the near-term period (2030–2040). With this model, we will have not only the current in-force policies but also the possible policies that may be needed in the future, and analyze all policies together to gain an understanding of the policy portfolio construction for different pathways and how the priority of each policy would change over the longer-term period (2040–2050).
## Workflow
The code consists of five main steps:
1. Data preparation:

   Import all the input data. Build a "Policy_evaluation" file for the following MCDA analysis based on the characteristics of the policies, the experts' opinions, the technologies each policy is intended to support, and the factual data of these technologies from the present to 2050.
2. MCDA conducting:

   Calculate the MCDA scores based on the "Policy_evaluation" file by choosing the TOPSIS method for both time periods (2030–2040 and 2040–2050).
3. Relevance Score calculation:

   Calculate the relevance score for each policy based on the comparison of the technology deployment levels between the EP2050+ scenario and the MIX, REMIX, and H2 scenarios, as well as the technologies each policy is intended to support.
4. Robustness analysis:

   Include both uncertainty analysis and sensitivity analysis. For the uncertainty analysis, consider uncertainties in the technology data, differences in experts' opinions, and the allocation of the policy support levels to different technologies, and use Monte Carlo simulation to run a sufficient number of iterations and observe the variation in the results. For the sensitivity analysis, only consider the sensitivity of the TOPSIS criteria weights. Analyze how much the weights can change without leading to a change in the ranking for both time periods.
5. Result visualization:

   Generate the final plots to present the relevance scores and TOPSIS scores, and generate the policy portfolio results along with the visualization of the robustness analysis results.
## Input data
1. Policy related data: Policy characteristic (Policy_data.csv), Experts' opinions (Criteria_Acceptance_Ins.csv, Criteria_Admin_Burden.csv, Criteria_Perceived_Equity.csv) ;
2. Technology related data: Technology factual data (Technology_data.csv).
3. Scenario data: The energy system results from the scenario simulation (Scenario_energy_production.csv, Scenario_installed_capacity.csv);
4. Others: Technology policy relationship (Technology_policy_matrix.csv), Experts' judgments to criteria weights (AHP_weight_input.csv, BWM_weight_input.csv).

## 1 Data Preparation
* Load all input files, verify structure and technology-name consistency.
* Build criteria data, each tested individually before combining.
  * Cost of carbon abatement: For each policy, take the technologies it supports (from Technology_policy_matrix), normalize their weights to sum to 1, then compute a weighted average of Cost_current (for 2030-2040) and Cost_2050 (for 2040-2050) from Technology_data.
  * Social Acceptance of technologies: Same logic as Cost, just a different source column. So this value is the SAME for both time periods.
  * Deployment difficulty:
    * For the EP2050+ baseline scenario, compute each technology's SHARE of
     its own sector's total production (so shares sum to 1 within a sector).
    * For each policy, take the weighted average of these shares across the
     technologies it supports (using the same normalized policy-technology
     weights as the cost or acceptance of technologies).
    * Same value for both time periods.
    * Interpretation: LOW share = technology is barely present in the baseline
     -> policy needs to work HARDER to promote it -> HIGH deployment difficulty.
     This direction (low value = harder) will be handled later by TOPSIS.
  * Cost gap: Cost_current minus Cost_2050. Only used in the SECOND TOPSIS run (2040-2050).
  * Acceptance of policy instrument, Perceived equity, Administrative burden (Qualitative expert-opinion criteria): Look up the policy's category (Instrument / Administrative_touchpoint / their combo) in a small reference table and pull the 'mode' value for main TOPSIS. Same value for BOTH time periods.
* Assemble two clean Policy_evaluation tables, ready for TOPSIS.
  * Period 1 (2030-2040): 6 criteria, using Cost_current
  * Period 2 (2040-2050): 7 criteria, using Cost_2050, PLUS Cost gap

In [1]:
# Data Import

import pandas as pd
import numpy as np

## Create file paths
DATA_DIR = "D:/Tansy/Master thesis/MCDA/data/final/input/"  # <-- change this to your folder

files = {
    "policy": "Policy_data.csv",
    "tech_policy_matrix": "Technology_policy_matrix.csv",
    "technology": "Technology_data.csv",
    "scenario_production": "Scenario_energy_production.csv",
    "scenario_capacity": "Scenario_installed_capacity.csv",
    "crit_acceptance_ins": "Criteria_Acceptance_Ins.csv",
    "crit_admin_burden": "Criteria_Admin_Burden.csv",
    "crit_perceived_equity": "Criteria_Perceived_Equity.csv",
    "AHP_input": "AHP_weight_input.csv",
    "BWM_input": "BWM_weight_input.csv"
}

## Load everything into a dictionary of DataFrames
data = {}
for key, filename in files.items():
    data[key] = pd.read_csv(DATA_DIR + filename)

## Sanity check on each table
print("=" * 70)
for key, df in data.items():
    print(f"[{key}]  ({files[key]})")
    print(f"  shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"  columns: {list(df.columns)}")
    n_missing = df.isna().sum().sum()
    print(f"  total missing values: {n_missing}")
    print("-" * 70)

[policy]  (Policy_data.csv)
  shape: 48 rows x 9 columns
  columns: ['Policy_ID', 'Status', 'Order', 'Name', 'Source_policies', 'Energy_sector', 'Promoting_technologies', 'Instrument', 'Administrative_touchpoint']
  total missing values: 0
----------------------------------------------------------------------
[tech_policy_matrix]  (Technology_policy_matrix.csv)
  shape: 48 rows x 20 columns
  columns: ['Policy_ID', 'Biomass_Boiler', 'Heat_Pump', 'Methane_Boiler', 'CHP_waste_heating', 'CHP_Methane_heating', 'CCGT_Ren_Methane', 'CHP_waste_electricity', 'CHP_Methane_electricity', 'Hydro_reservoir', 'Hydro_run_of', 'PV_Roof', 'Pumped_Hydro', 'Wind', 'CCGT_Gas_DACCS', 'Alpine_PV', 'Heavy_EV', 'Heavy_FCEV', 'Light_EV', 'Light_FCEV']
  total missing values: 768
----------------------------------------------------------------------
[technology]  (Technology_data.csv)
  shape: 19 rows x 4 columns
  columns: ['Technology', 'Acceptance', 'Cost_current', 'Cost_2050']
  total missing values: 0
----

In [2]:
# Build Policy_evaluation table

## Criterion 1: Cost of carbon abatement (EUR/tCO2eq)

### Take the Technology_policy_matrix and normalize each row so its non-missing weights sum to 1. Rows that are all-NaN stay all-NaN.
def normalize_weights(matrix_df, id_col="Policy_ID"):
    tech_cols = [c for c in matrix_df.columns if c != id_col]
    weights = matrix_df.set_index(id_col)[tech_cols]
    row_sums = weights.sum(axis=1, skipna=True)
    normalized = weights.div(row_sums, axis=0)
    return normalized  # index = Policy_ID, columns = technologies, values = normalized weights

### For each policy, compute sum( weight_i * cost_i )  over technologies i the policy supports
def compute_weighted_cost(normalized_weights, tech_df, cost_col):
    # Build a lookup: technology name -> cost value
    cost_lookup = tech_df.set_index("Technology")[cost_col]
    # Reindex so column order in normalized_weights matches cost_lookup order， any tech name mismatch becomes a NaN column。
    aligned_costs = cost_lookup.reindex(normalized_weights.columns)
    # Weighted sum
    weighted_cost = normalized_weights.mul(aligned_costs, axis=1).sum(axis=1, skipna=True)
    return weighted_cost

### Run it
weights = normalize_weights(data["tech_policy_matrix"])

### Sanity check
tech_names_in_data = set(data["technology"]["Technology"])
tech_names_in_matrix = set(weights.columns)
missing_names = tech_names_in_matrix - tech_names_in_data
if missing_names:
    print(f"WARNING: these matrix columns have NO match in Technology_data.csv: {missing_names}")
else:
    print("All technology names in the matrix match Technology_data.csv")

cost_2030_2040 = compute_weighted_cost(weights, data["technology"], "Cost_current")
cost_2040_2050 = compute_weighted_cost(weights, data["technology"], "Cost_2050")

### Print results
print("\nFirst 5 policies - Cost of carbon abatement (EUR/tCO2eq):")
preview = pd.DataFrame({
    "Cost_2030_2040": cost_2030_2040,
    "Cost_2040_2050": cost_2040_2050,
}).head()
print(preview)

### Check NaN results
n_nan = cost_2030_2040.isna().sum()
print(f"\nPolicies with missing Cost result: {n_nan} out of {len(cost_2030_2040)}")

## Criterion 2: Social Acceptance of technologies

### Run it again
acceptance_score = compute_weighted_cost(weights, data["technology"], "Acceptance")
acceptance_2030_2040 = acceptance_score
acceptance_2040_2050 = acceptance_score

print("\nFirst 5 policies - Social Acceptance of technologies (same both periods):")
preview2 = pd.DataFrame({
    "Acceptance_2030_2040": acceptance_2030_2040,
    "Acceptance_2040_2050": acceptance_2040_2050,
}).head()
print(preview2)

n_nan2 = acceptance_score.isna().sum()
print(f"\nPolicies with missing Acceptance result: {n_nan2} out of {len(acceptance_score)}")

All technology names in the matrix match Technology_data.csv

First 5 policies - Cost of carbon abatement (EUR/tCO2eq):
           Cost_2030_2040  Cost_2040_2050
Policy_ID                                
I_01          4749.681785     2950.054383
N_02          2560.649435     1347.986975
I_03          2573.741835     1648.083733
I_04          4749.681785     2950.054383
I_05          2560.649435     1347.986975

Policies with missing Cost result: 0 out of 48

First 5 policies - Social Acceptance of technologies (same both periods):
           Acceptance_2030_2040  Acceptance_2040_2050
Policy_ID                                            
I_01                      0.714                 0.714
N_02                      0.235                 0.235
I_03                      0.660                 0.660
I_04                      0.714                 0.714
I_05                      0.235                 0.235

Policies with missing Acceptance result: 0 out of 48


In [3]:
## Criterion 3: Deployment level under baseline pathway (aka deployment difficulty)

### Compute each technology's share of its sector's EP2050+ total
sector_totals = data["scenario_production"].groupby("Sector")["EP2050+"].transform("sum")
data["scenario_production"]["EP2050_share"] = data["scenario_production"]["EP2050+"] / sector_totals

print("\nSanity check - shares should sum to 1.0 within each sector:")
print(data["scenario_production"].groupby("Sector")["EP2050_share"].sum())

### Weighted average of these shares across each policy's supported technologies (reuse the compute_weighted_cost function)
deployment_level = compute_weighted_cost(weights, data["scenario_production"], "EP2050_share")
deployment_level_2030_2040 = deployment_level
deployment_level_2040_2050 = deployment_level

print("\nFirst 5 policies - Deployment level (baseline share):")
print(deployment_level.head())

n_nan3 = deployment_level.isna().sum()
print(f"\nPolicies with missing Deployment level result: {n_nan3} out of {len(deployment_level)}")

### Criterion 7: Cost gap
cost_gap = cost_2030_2040 - cost_2040_2050

print("\nFirst 5 policies - Cost gap (EUR/tCO2eq saved by 2050):")
print(cost_gap.head())


Sanity check - shares should sum to 1.0 within each sector:
Sector
Electricity    1.0
Heating        1.0
Transport      1.0
Name: EP2050_share, dtype: float64

First 5 policies - Deployment level (baseline share):
Policy_ID
I_01    0.196376
N_02    0.250000
I_03    0.033813
I_04    0.196376
I_05    0.250000
dtype: float64

Policies with missing Deployment level result: 0 out of 48

First 5 policies - Cost gap (EUR/tCO2eq saved by 2050):
Policy_ID
I_01    1799.627402
N_02    1212.662460
I_03     925.658101
I_04    1799.627402
I_05    1212.662460
dtype: float64


In [4]:
## Criteria 4-6: Acceptance of policy instrument, Perceived equity, Administrative burden (Qualitative expert-opinion criteria)

### Build the lookup function
def lookup_by_category(policy_df, category_col, ref_df, ref_category_col, value_col="mode"):
    # Standardize categories' names on BOTH sides before matching
    policy_key = policy_df[category_col].str.strip().str.lower()
    ref_key = ref_df[ref_category_col].str.strip().str.lower()

    # Build a lookup: normalized category name -> mode value
    value_lookup = pd.Series(ref_df[value_col].values, index=ref_key)

    result = policy_key.map(value_lookup)
    result.index = policy_df["Policy_ID"]
    return result

policy_df = data["policy"]

### Criterion 4: Acceptance of policy instrument
acceptance_ins = lookup_by_category(
    policy_df, "Instrument",
    data["crit_acceptance_ins"], "Instrument",
)

# Criterion 5: Administrative burden
admin_burden = lookup_by_category(
    policy_df, "Administrative_touchpoint",
    data["crit_admin_burden"], "Administrative_touchpoint",
)

# Criterion 6: Perceived equity
policy_df = policy_df.copy()
policy_df["Instrument_Administrative_touchpoint"] = (
    policy_df["Instrument"] + "_" + policy_df["Administrative_touchpoint"]
)
perceived_equity = lookup_by_category(
    policy_df, "Instrument_Administrative_touchpoint",
    data["crit_perceived_equity"], "Instrument_Administrative_touchpoint",
)

### Check for any unmatched (NaN) results
for name, series in [
    ("Acceptance of policy instrument", acceptance_ins),
    ("Administrative burden", admin_burden),
    ("Perceived equity", perceived_equity),
]:
    n_missing = series.isna().sum()
    status = "all matched" if n_missing == 0 else f"{n_missing} unmatched!"
    print(f"{name}: {status}")

print("\nFirst 5 policies - all 3 qualitative criteria:")
print(pd.DataFrame({
    "Acceptance_Ins": acceptance_ins,
    "Admin_Burden": admin_burden,
    "Perceived_Equity": perceived_equity,
}).head())

Acceptance of policy instrument: all matched
Administrative burden: all matched
Perceived equity: all matched

First 5 policies - all 3 qualitative criteria:
           Acceptance_Ins  Admin_Burden  Perceived_Equity
Policy_ID                                                
I_01                    1             3                 4
N_02                    1             3                 4
I_03                    4             3                 2
I_04                    1             3                 4
I_05                    1             3                 4


In [5]:
# Assemble Policy_evaluation tables

policy_evaluation_2030_2040 = pd.DataFrame({
    "Cost_carbon_abatement": cost_2030_2040,
    "Social_acceptance_tech": acceptance_2030_2040,
    "Deployment_difficulty": deployment_level_2030_2040,
    "Acceptance_policy_instrument": acceptance_ins,
    "Perceived_equity": perceived_equity,
    "Administrative_burden": admin_burden,
})

policy_evaluation_2040_2050 = pd.DataFrame({
    "Cost_carbon_abatement": cost_2040_2050,
    "Social_acceptance_tech": acceptance_2040_2050,
    "Deployment_difficulty": deployment_level_2040_2050,
    "Acceptance_policy_instrument": acceptance_ins,
    "Perceived_equity": perceived_equity,
    "Administrative_burden": admin_burden,
    "Cost_gap": cost_gap,  # extra column, not used in TOPSIS yet
})

print("\nPolicy_evaluation_2030_2040 (first 5 rows):")
print(policy_evaluation_2030_2040.head())
print(f"\nShape: {policy_evaluation_2030_2040.shape}")
print(f"Any missing values? {policy_evaluation_2030_2040.isna().sum().sum()}")

print("\n" + "-" * 70)
print("\nPolicy_evaluation_2040_2050 (first 5 rows):")
print(policy_evaluation_2040_2050.head())
print(f"\nShape: {policy_evaluation_2040_2050.shape}")
print(f"Any missing values? {policy_evaluation_2040_2050.isna().sum().sum()}")

# Save to CSV
policy_evaluation_2030_2040.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/intermediate/Policy_evaluation_2030_2040.csv",
    index=True)  # <-- change this to your folder
policy_evaluation_2040_2050.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/intermediate/policy_evaluation_2040_2050.csv",
    index=True)  # <-- change this to your folder
print("\n✅ Saved: Policy_evaluation_2030_2040.csv, Policy_evaluation_2040_2050.csv")


Policy_evaluation_2030_2040 (first 5 rows):
           Cost_carbon_abatement  Social_acceptance_tech  \
Policy_ID                                                  
I_01                 4749.681785                   0.714   
N_02                 2560.649435                   0.235   
I_03                 2573.741835                   0.660   
I_04                 4749.681785                   0.714   
I_05                 2560.649435                   0.235   

           Deployment_difficulty  Acceptance_policy_instrument  \
Policy_ID                                                        
I_01                    0.196376                             1   
N_02                    0.250000                             1   
I_03                    0.033813                             4   
I_04                    0.196376                             1   
I_05                    0.250000                             1   

           Perceived_equity  Administrative_burden  
Policy_ID         

## 2 MCDA conducting

### Decide weights of the criteria (Method: AHP or BWM)
ONE set of weights is used for both time periods (not two separate sets). Since Period 1 has 6 criteria and Period 2 has 7 (plus Cost_gap), we derive weights on the FULL 7-criteria set, then for Period 1 we drop the Cost_gap weight and rescale the remaining 6.

In the two input files for weights, the cells both scale from 1 to 9, answering "how much more important is criterion i than criterion j?"

(1=equal, 3=moderate, 5=strong, 7=very strong, 9=extreme; use 2/4/6/8 for in-between)

* AHP_weight_input.csv
  * Input: 7x7 matrix, Matrix must be reciprocal (cell [j,i] = 1 / cell [i,j]), diagonal = 1.
  * Output: weight vector (sums to 1) + Consistency Ratio (CR). CR < 0.10 is considered acceptable.
* BWM_weight_input.csv
  * Input: one row per criterion, with Is_Best/Is_Worst flags and Best_to_Criterion / Criterion_to_Worst comparison values.
  * Output: weight vector (sums to 1) + consistency indicator (xi, closer to 0 = more consistent).

### TOPSIS (both time periods)
Core idea: the best policy is the one CLOSEST to an imaginary "ideal best" combination of all criteria, and FARTHEST from an imaginary "ideal worst".
* Criteria directions:
  * Cost_carbon_abatement          - lower better (negative)
  * Social_acceptance_tech         - lower better (negative)
  * Deployment_difficulty          - lower better (negative)
  * Acceptance_policy_instrument   - higher better (positive)
  * Perceived_equity               - higher better (positive)
  * Administrative_burden          - lower better (negative)
  * Cost_gap (Period 2 only)       - lower better (negative)

In [6]:
# Criteria weights (AHP or BWM)

import scipy.optimize
from scipy.optimize import linprog

## AHP function

## Random Index table for AHP consistency check
RANDOM_INDEX = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}

### AHP weights calculation function
def ahp_weights(pairwise_matrix, criteria_names):

    n = pairwise_matrix.shape[0]
    assert pairwise_matrix.shape == (n, n)  # Matrix must be square

    # Normalize the Pairwise Comparison Matrix and Calculate Criterion Weights
    col_sums = pairwise_matrix.sum(axis=0)
    normalized = pairwise_matrix / col_sums
    weights = normalized.mean(axis=1)

    # Calculate the Consistency Ratio (CR)
    weighted_sum = pairwise_matrix @ weights
    lambda_max = (weighted_sum / weights).mean()
    CI = (lambda_max - n) / (n - 1) if n > 1 else 0.0
    RI = RANDOM_INDEX.get(n, 1.49)
    CR = CI / RI if RI > 0 else 0.0

    weights_series = pd.Series(weights, index=criteria_names, name="AHP_weight")
    return weights_series, CR

## BWM function

### BWM weights calculation function
def bwm_weights(criteria_names, best_index, worst_index, best_to_others, others_to_worst):

    n = len(criteria_names)
    n_vars = n + 1
    xi_idx = n

    A_ub, b_ub = [], []
    for j in range(n):
        a_Bj = best_to_others[j]
        row1 = [0.0] * n_vars; row1[best_index] += 1; row1[j] -= a_Bj; row1[xi_idx] = -1
        A_ub.append(row1); b_ub.append(0.0)
        row2 = [0.0] * n_vars; row2[best_index] -= 1; row2[j] += a_Bj; row2[xi_idx] = -1
        A_ub.append(row2); b_ub.append(0.0)

        a_jW = others_to_worst[j]
        row3 = [0.0] * n_vars; row3[j] += 1; row3[worst_index] -= a_jW; row3[xi_idx] = -1
        A_ub.append(row3); b_ub.append(0.0)
        row4 = [0.0] * n_vars; row4[j] -= 1; row4[worst_index] += a_jW; row4[xi_idx] = -1
        A_ub.append(row4); b_ub.append(0.0)

    A_eq = [[1.0] * n + [0.0]]
    b_eq = [1.0]
    bounds = [(0, 1)] * n + [(0, None)]
    c = [0.0] * n + [1.0]

    result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not result.success:
        raise RuntimeError(f"BWM optimization failed: {result.message}")

    weights = result.x[:n]
    xi = result.x[xi_idx]
    weights_series = pd.Series(weights, index=criteria_names, name="BWM_weight")
    return weights_series, xi

### Call the BWM function
def bwm_weights_from_csv(df):

    criteria_names = df["Criterion"].tolist()
    best_index = df.index[df["Is_Best"] == 1][0]
    worst_index = df.index[df["Is_Worst"] == 1][0]
    best_to_others = df["Best_to_Criterion"].tolist()
    others_to_worst = df["Criterion_to_Worst"].tolist()
    return bwm_weights(criteria_names, best_index, worst_index, best_to_others, others_to_worst)

## Code for AHP - Create a Subset from All Criteria and Renormalize Criterion Weights
def subset_and_renormalize(weights_series, subset_criteria):
    subset = weights_series.loc[subset_criteria]
    return subset / subset.sum()

## Load the criteria set from both weight input files
criteria_p2 = ["Cost_carbon_abatement", "Social_acceptance_tech", "Deployment_difficulty",
               "Acceptance_policy_instrument", "Perceived_equity", "Administrative_burden",
               "Cost_gap"]
criteria_p1 = criteria_p2[:-1]  # everything except Cost_gap

## Weight calculation

### AHP 2040-2050
print(f"\n--- AHP: {len(criteria_p2)}-criteria weights")
ahp_full_weights, ahp_cr = ahp_weights(data["AHP_input"].iloc[:,1:].values, data["AHP_input"].columns[1:].tolist())
print(ahp_full_weights)
print(f"Consistency Ratio: {ahp_cr:.4f}  (should be < 0.10 to be considered acceptable)")


### BWM 2040-2050
print(f"\n--- BWM: {len(criteria_p2)}-criteria weights")
bwm_full_weights_p2, bwm_xi_p2 = bwm_weights_from_csv(data["BWM_input"])
print(bwm_full_weights_p2)
print(f"Consistency indicator (xi): {bwm_xi_p2:.4f}  (closer to 0 = more consistent)")

### BWM 2030-2040
print(f"\n--- BWM: {len(criteria_p1)}-criteria weights")
bwm_full_weights_p1, bwm_xi_p1 = bwm_weights_from_csv(data["BWM_input"][:-1])
print(bwm_full_weights_p1)
print(f"Consistency indicator (xi): {bwm_xi_p1:.4f}  (closer to 0 = more consistent)")

## Choose which method feeds TOPSIS, AHP/BWM

### weights_for_topsis = bwm_full_weights   # Shift to this line if using AHP

## Derive the Period 1 (6-criteria) weight vector
### weights_p1 = subset_and_renormalize(weights_for_topsis, criteria_p1)           # Shift to this line if using AHP
weights_p1 = bwm_full_weights_p1
weights_p2 = bwm_full_weights_p2  # already matches all 7 criteria

print(f"\n--- Final weights used in TOPSIS ---")
print("Period 1 (6 criteria):")
print(weights_p1)
print(f"Sum: {weights_p1.sum():.4f}  (should be 1.0)")
print("\nPeriod 2 (7 criteria):")
print(weights_p2)
print(f"Sum: {weights_p2.sum():.4f}  (should be 1.0)")


--- AHP: 7-criteria weights
Cost_carbon_abatement           0.142857
Social_acceptance_tech          0.142857
Deployment_difficulty           0.142857
Acceptance_policy_instrument    0.142857
Perceived_equity                0.142857
Administrative_burden           0.142857
Cost_gap                        0.142857
Name: AHP_weight, dtype: float64
Consistency Ratio: 0.0000  (should be < 0.10 to be considered acceptable)

--- BWM: 7-criteria weights
Cost_carbon_abatement           0.154930
Social_acceptance_tech          0.260563
Deployment_difficulty           0.154930
Acceptance_policy_instrument    0.042254
Perceived_equity                0.154930
Administrative_burden           0.077465
Cost_gap                        0.154930
Name: BWM_weight, dtype: float64
Consistency indicator (xi): 0.0493  (closer to 0 = more consistent)

--- BWM: 6-criteria weights
Cost_carbon_abatement           0.183333
Social_acceptance_tech          0.308333
Deployment_difficulty           0.183333
Acceptan

In [7]:
# TOPSIS

## Build TOPSIS function

def topsis(decision_df, weights, directions):

    cols = decision_df.columns.tolist()
    w = np.array([weights[c] for c in cols])
    dirs = [directions[c] for c in cols]

    X = decision_df.values.astype(float)

    # Step 1: vector normalization (each column divided by its Euclidean norm)
    norms = np.sqrt((X ** 2).sum(axis=0))
    X_norm = X / norms

    # Step 2: apply weights
    X_weighted = X_norm * w

    # Step 3: ideal best / ideal worst (nadir) per column, by direction
    ideal_best = np.where(
        np.array(dirs) == "positive",
        X_weighted.max(axis=0),
        X_weighted.min(axis=0),
    )
    ideal_worst = np.where(
        np.array(dirs) == "positive",
        X_weighted.min(axis=0),
        X_weighted.max(axis=0),
    )

    # Step 4: Euclidean distances
    dist_to_ideal = np.sqrt(((X_weighted - ideal_best) ** 2).sum(axis=1))
    dist_to_nadir = np.sqrt(((X_weighted - ideal_worst) ** 2).sum(axis=1))

    # Step 5: TOPSIS Score (closeness coefficient)
    topsis_score = dist_to_nadir / (dist_to_ideal + dist_to_nadir)

    result = pd.DataFrame({
        "TOPSIS Score": topsis_score,
        "Distance to Ideal": dist_to_ideal,
        "Distance to Nadir": dist_to_nadir,
    }, index=decision_df.index)

    # Add one Score column per criterion
    for i, c in enumerate(cols):
        result[f"{c} Score"] = X_norm[:, i]

    # Step 6: rank (1 = best).
    result["Rank"] = result["TOPSIS Score"].rank(ascending=False).astype(int)

    # Final column order, as requested
    score_cols = [f"{c} Score" for c in cols]
    result = result[["Rank", "TOPSIS Score", "Distance to Ideal", "Distance to Nadir"] + score_cols]
    return result


## Criteria directions
directions_p1 = {
    "Cost_carbon_abatement": "negative",
    "Social_acceptance_tech": "negative",
    "Deployment_difficulty": "negative",
    "Acceptance_policy_instrument": "positive",
    "Perceived_equity": "positive",
    "Administrative_burden": "negative",
}
directions_p2 = {**directions_p1, "Cost_gap": "negative"}

## Run TOPSIS
### Period 1 (2030-2040): 6 criteria
topsis_p1 = topsis(policy_evaluation_2030_2040, weights_p1, directions_p1)

print("\n--- TOPSIS results: Period 1 (2030-2040) (first 5 rows) ---")
print(topsis_p1.head())
print(f"\nTotal policies ranked: {len(topsis_p1)} (should be 48)")

### Period 2 (2040-2050): 7 criteria (incl. Cost_gap)
topsis_p2 = topsis(policy_evaluation_2040_2050, weights_p2, directions_p2)

print("\n--- TOPSIS results: Period 2 (2040-2050) (first 5 rows) ---")
print(topsis_p2.head())
print(f"\nTotal policies ranked: {len(topsis_p2)} (should be 48)")

# ---- Save results -------------------------------------------------------------
topsis_p1.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/TOPSIS_results_2030_2040.csv", index=True)  # <-- change this to your folder
topsis_p2.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/TOPSIS_results_2040_2050.csv", index=True)  # <-- change this to your folder
print("\n Saved: TOPSIS_results_2030_2040.csv, TOPSIS_results_2040_2050.csv")


--- TOPSIS results: Period 1 (2030-2040) (first 5 rows) ---
           Rank  TOPSIS Score  Distance to Ideal  Distance to Nadir  \
Policy_ID                                                             
I_01         38      0.590527           0.064735           0.093358   
N_02         12      0.761555           0.034526           0.110272   
I_03         25      0.673177           0.054230           0.111700   
I_04         38      0.590527           0.064735           0.093358   
I_05         12      0.761555           0.034526           0.110272   

           Cost_carbon_abatement Score  Social_acceptance_tech Score  \
Policy_ID                                                              
I_01                          0.184879                      0.179037   
N_02                          0.099672                      0.058927   
I_03                          0.100182                      0.165496   
I_04                          0.184879                      0.179037   
I_05     

## 3 Relevance Score calculation
### Relevance Score
For each of the 3 alternative scenarios (MIX, REMIX, H2), measure how much each technology needs to increase/decrease relative to the EP2050+ baseline, then use the technology-policy matrix to turn that into a per-policy relevance score for each scenario.

* Formula per technology: relevance_tech = (Scenario_value - EP2050+_value) / EP2050+_sector_total
  * Positive = scenario needs MORE of this technology than the baseline
   * Negative = scenario needs LESS
   * ~0 = no change of this technology under this scenario

* Per-policy score: Relevance Score = weighted average of relevance_tech across the policy's supported technologies
 (same normalized Technology_policy_matrix weights used throughout this pipeline).

### Classify into 5 groups
To classify each policy's relevance score (per scenario) into 5 groups, use Ckmeans.1d.dp (Wang & Song, 2011), the exact optimal 1D clustering method, which finds group boundaries that minimize within-group variance.

* Groups: Strong negative | Weak negative | Neutral | Weak positive | Strong positive

Make sure the relevance score 0 always falls inside Neutral group, since this score implicates that the policy doesn't need to be adjusted in the alternative scenario.


In [8]:
# Relevance score

## technology-level relevance (increase/decrease vs EP2050+)
sector_totals_ep2050 = data["scenario_production"].groupby("Sector")["EP2050+"].transform("sum")

alt_scenarios = ["MIX", "REMIX", "H2"]
for scen in alt_scenarios:
    data["scenario_production"][f"relevance_{scen}"] = (
        (data["scenario_production"][scen] - data["scenario_production"]["EP2050+"]) / sector_totals_ep2050
    )

### Save and print intermediate results
relevance_tech = data["scenario_production"][["Sector", "Technology"] + [f"relevance_{s}" for s in alt_scenarios]]
relevance_tech.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/intermediate/Technology_relevance.csv", index=False)  # <-- change this to your folder
print("\nTechnology-level relevance (first 5 rows):")
print(relevance_tech.head())
print("\n Saved: Technology_relevance.csv")

## Weight average across each policy's supported technologies

tech_relevance_lookup = data["scenario_production"].set_index("Technology")[
    [f"relevance_{s}" for s in alt_scenarios]
].reset_index()

policy_relevance = pd.DataFrame(index=weights.index)
for scen in alt_scenarios:
    policy_relevance[f"Relevance_{scen}"] = compute_weighted_cost(
        weights, tech_relevance_lookup, f"relevance_{scen}"
    )

print("\nPolicy-level relevance scores (first 5 policies):")
print(policy_relevance.head())

n_missing = policy_relevance.isna().sum().sum()
print(f"\nAny missing values? {n_missing}")


## Classify into 5 groups by Ckmeans.1d.dp

### Build Ckmeans.1d.dp fucntion
def ckmeans_1d_dp(values, n_classes):

    data = sorted(values)
    n = len(data)

    lower_class_limits = [[0] * (n_classes + 1) for _ in range(n + 1)]
    variance_combinations = [[float("inf")] * (n_classes + 1) for _ in range(n + 1)]
    variance_combinations[1][1] = 0.0

    for i in range(2, n + 1):
        sum_, sum_squares, w = 0.0, 0.0, 0.0
        for m in range(1, i + 1):
            lower_class_limit = i - m + 1
            val = data[lower_class_limit - 1]
            sum_ += val
            sum_squares += val * val
            w += 1
            variance = sum_squares - (sum_ * sum_) / w
            i4 = lower_class_limit - 1
            if i4 != 0:
                for j in range(2, n_classes + 1):
                    candidate = variance + variance_combinations[i4][j - 1]
                    if variance_combinations[i][j] >= candidate:
                        lower_class_limits[i][j] = lower_class_limit
                        variance_combinations[i][j] = candidate
        lower_class_limits[i][1] = 1
        variance_combinations[i][1] = sum_squares - (sum_ * sum_) / w

    # Backtrack through the DP table to recover the actual breakpoint values
    k = n
    kclass = [0] * (n_classes + 1)
    kclass[n_classes] = data[-1]
    kclass[0] = data[0]
    count_num = n_classes
    while count_num >= 2:
        idx = int(lower_class_limits[k][count_num] - 2)
        kclass[count_num - 1] = data[idx]
        k = int(lower_class_limits[k][count_num] - 1)
        count_num -= 1

    return kclass

### Function to make sure 0 is inside the "Neutral" group
def enforce_zero_in_neutral(breaks, neutral_index):

    adjusted = list(breaks)
    lower_idx, upper_idx = neutral_index, neutral_index + 1
    if adjusted[lower_idx] > 0:
        adjusted[lower_idx] = 0.0
    if adjusted[upper_idx] < 0:
        adjusted[upper_idx] = 0.0
    return adjusted

### Build labeling function for each group after classification
def ckmeans_classify(values, n_classes, labels, guarantee_zero_neutral=True):

    breaks = ckmeans_1d_dp(values.tolist(), n_classes)

    if guarantee_zero_neutral and "Neutral" in labels:
        neutral_index = labels.index("Neutral")
        breaks = enforce_zero_in_neutral(breaks, neutral_index)

        lower_bound = breaks[neutral_index]
        upper_bound = breaks[neutral_index + 1]

        def assign(v):
            if lower_bound <= v <= upper_bound:
                return labels[neutral_index]
            if v < lower_bound:
                for i in range(neutral_index):
                    if v <= breaks[i + 1]:
                        return labels[i]
                return labels[neutral_index - 1]
            else:
                for i in range(neutral_index + 1, n_classes):
                    if v <= breaks[i + 1]:
                        return labels[i]
                return labels[-1]
    else:
        def assign(v):
            for i in range(n_classes):
                if v <= breaks[i + 1]:
                    return labels[i]
            return labels[-1]

    return values.apply(assign), breaks


### Apply Ckmeans.1d.dp classification to each scenario's relevance score
group_labels = ["Strong negative", "Weak negative", "Neutral", "Weak positive", "Strong positive"]

for scen in alt_scenarios:
    col = f"Relevance_{scen}"
    group_col = f"Group_{scen}"
    labels_series, breaks = ckmeans_classify(policy_relevance[col], 5, group_labels)
    policy_relevance[group_col] = labels_series
    print(f"\n--- Ckmeans.1d.dp breaks for {scen} ---")
    print("Breakpoints:", [round(b, 4) for b in breaks])
    print(policy_relevance[group_col].value_counts().reindex(group_labels))

print("\nFull relevance table (first 10 policies):")
print(policy_relevance.head(10))

policy_relevance.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/Policy_relevance_scores.csv", index=True)  # <-- change this to your folder
print("\n Saved: Policy_relevance_scores.csv")


Technology-level relevance (first 5 rows):
    Sector           Technology  relevance_MIX  relevance_REMIX  relevance_H2
0  Heating       Biomass_Boiler      -0.457983        -0.203081      0.030812
1  Heating            Heat_Pump       0.537815         0.236695     -0.257703
2  Heating       Methane_Boiler       0.000000         0.000000      0.226891
3  Heating    CHP_waste_heating       0.000000         0.000000      0.000000
4  Heating  CHP_Methane_heating      -0.036415        -0.036415      0.000000

 Saved: Technology_relevance.csv

Policy-level relevance scores (first 5 policies):
           Relevance_MIX  Relevance_REMIX  Relevance_H2
Policy_ID                                              
I_01            0.009204        -0.000040 -6.938894e-18
N_02            0.000000         0.000000  1.734723e-17
I_03           -0.033813        -0.033813  0.000000e+00
I_04            0.009204        -0.000040 -6.938894e-18
I_05            0.000000         0.000000  1.734723e-17

Any missin

## 4 Robustness analysis
### Weight Sensitivity Analysis
For each criterion, find how far its weight can move (up and down), while proportionally rescaling all other criteria weights to preserve their relative ratio to each other, before the final rank order of all 48 policies changes.

To achieve this, do a fine-grained scan of possible weights (from 0 to 1) for each criterion, rescaling the rest each time, re-running TOPSIS, and comparing the resulting Rank column to the baseline. Report the widest contiguous range around the original weight where ranks stay identical.
### Uncertainty Analysis
1. Compensatory relevance score

   Add the Relevance score (per alternative scenario: MIX, REMIX, H2) as an EXTRA criterion in TOPSIS, weighted at 0.5 with all original criteria weights rescaled down proportionally so everything still sums to 1.
   * Direction: positive
   * Produce 6 TOPSIS runs total: 2 time periods x 3 alternative scenarios.
2. Monte Carlo Uncertainty Analysis

   There are four perturbed inputs in the Monte Carlo analysis. Each iteration randomly perturbs 4 uncertain inputs, rebuilds the entire Policy_evaluation tables + relevance scores from scratch using those perturbed inputs, then re-runs TOPSIS (both periods) and Ckmeans (all 3 scenarios). (In TOPSIS, weights stay FIXED across iterations, only the underlying CRITERIA VALUES vary.) Finally, save Rank + TOPSIS Score, and Relevance + Group.

   The 4 perturbed inputs are:

   * Technology_policy_matrix (per policy, per supported technology):
     * Mature technologies (Biomass_Boiler, Methane_Boiler, CHP_Methane_heating, CCGT_Ren_Methane, CHP_Methane_electricity): uniform[current, current*1.2]
     * PV_Roof, ONLY in policies supporting EXACTLY {PV_Roof, Alpine_PV}: uniform[current, current*1.5]
     * Light_EV, in ANY policy where it appears alongside other technologies: uniform[current, ccurrent*1.5]
     * CHP_Methane_electricity: NOT sampled independently - always forced to CHP_Methane_heating_value * (0.44/0.46), even if that falls outside its own +0.5 window
     * Everything else: uniform[current]
     * Policies with only 1 supported technology: skipped (a single value always normalizes to 1.0)
     * After perturbing, each policy's row is renormalized to sum to 1
   * Cost_current and Cost_2050 (per technology)
      * each independently sampled from a UNIFORM distribution in [0.8x, 1.2x] of the original value
   * Cost_gap
      * recomputed as (perturbed Cost_current - perturbed Cost_2050), weighted-averaged using this iteration's perturbed technology-policy matrix
   * Qualitative criteria (Acceptance_Ins, Admin_Burden, Perceived_Equity)
      * each sampled from a DISCRETE UNIFORM distribution over the integers in [low, high] (every whole number equally likely)

   NOT perturbed
   * Social_acceptance_tech and Deployment_difficulty use their ORIGINAL technology-level values
   * Recompute each iteration using the iteration's perturbed matrix, since the technology-policy matrix affects every matrix-weighted criterion, not just cost.

In [9]:
# Weight Sensitivity Analysis

## Rescale function for the non-targeting criteria
def rescale_others(weights_series, target_criterion, target_weight):

    others = weights_series.drop(target_criterion)
    original_others_sum = others.sum()
    scale_factor = (1 - target_weight) / original_others_sum
    new_weights = others * scale_factor
    new_weights[target_criterion] = target_weight
    return new_weights.reindex(weights_series.index)

## Function for finding the weight sensitivity range
def find_sensitivity_range(decision_df, base_weights, directions, target_criterion, baseline_ranks, step=0.005):

    current_w = base_weights[target_criterion]

    # Scan upward
    max_w = current_w
    w = current_w
    while w + step <= 1.0:
        w += step
        trial_weights = rescale_others(base_weights, target_criterion, w)
        trial_result = topsis(decision_df, trial_weights, directions)
        if not trial_result["Rank"].equals(baseline_ranks):
            break
        max_w = w

    # Scan downward
    min_w = current_w
    w = current_w
    while w - step >= 0.0:
        w -= step
        trial_weights = rescale_others(base_weights, target_criterion, w)
        trial_result = topsis(decision_df, trial_weights, directions)
        if not trial_result["Rank"].equals(baseline_ranks):
            break
        min_w = w

    return min_w, max_w

## Function to calculate the range for each criterion's weight
def run_sensitivity_analysis(decision_df, base_weights, directions, period_label):
    baseline_result = topsis(decision_df, base_weights, directions)
    baseline_ranks = baseline_result["Rank"]

    rows = []
    for criterion in base_weights.index:
        min_w, max_w = find_sensitivity_range(
            decision_df, base_weights, directions, criterion, baseline_ranks
        )
        rows.append({
            "Period": period_label,
            "Criterion": criterion,
            "Current_Weight": base_weights[criterion],
            "Min_Weight_No_Rank_Change": min_w,
            "Max_Weight_No_Rank_Change": max_w,
            "Allowed_Decrease": base_weights[criterion] - min_w,
            "Allowed_Increase": max_w - base_weights[criterion],
        })
        print(f"  [{period_label}] {criterion}: current={base_weights[criterion]:.4f}, "
              f"range=[{min_w:.4f}, {max_w:.4f}]")
    return pd.DataFrame(rows)

## Print and save the sensitivity analysis results
print("\n--- Period 1 (2030-2040) sensitivity ---")
sens_p1 = run_sensitivity_analysis(policy_evaluation_2030_2040, weights_p1, directions_p1, "2030_2040")

print("\n--- Period 2 (2040-2050) sensitivity ---")
sens_p2 = run_sensitivity_analysis(policy_evaluation_2040_2050, weights_p2, directions_p2, "2040_2050")

sensitivity_results = pd.concat([sens_p1, sens_p2], ignore_index=True)
sensitivity_results.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/Weight_sensitivity_results.csv", index=False)  # <-- change this to your folder
print("\n Saved: Weight_sensitivity_results.csv")
print(sensitivity_results)


--- Period 1 (2030-2040) sensitivity ---
  [2030_2040] Cost_carbon_abatement: current=0.1833, range=[0.1833, 0.1833]
  [2030_2040] Social_acceptance_tech: current=0.3083, range=[0.3083, 0.3083]
  [2030_2040] Deployment_difficulty: current=0.1833, range=[0.1833, 0.1833]
  [2030_2040] Acceptance_policy_instrument: current=0.0500, range=[0.0500, 0.0900]
  [2030_2040] Perceived_equity: current=0.1833, range=[0.1783, 0.1833]
  [2030_2040] Administrative_burden: current=0.0917, range=[0.0917, 0.0967]

--- Period 2 (2040-2050) sensitivity ---
  [2040_2050] Cost_carbon_abatement: current=0.1549, range=[0.1549, 0.1549]
  [2040_2050] Social_acceptance_tech: current=0.2606, range=[0.2606, 0.2606]
  [2040_2050] Deployment_difficulty: current=0.1549, range=[0.1549, 0.1549]
  [2040_2050] Acceptance_policy_instrument: current=0.0423, range=[0.0023, 0.0473]
  [2040_2050] Perceived_equity: current=0.1549, range=[0.1549, 0.1549]
  [2040_2050] Administrative_burden: current=0.0775, range=[0.0725, 0.0775

In [10]:
# Compensatory relevance score

relevance_df = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/Policy_relevance_scores.csv", index_col=0)    # <-- change this to your folder
alt_scenarios = ["MIX", "REMIX", "H2"]

## Add weight for relevance score and rescale
def add_relevance_weight(base_weights, relevance_weight=0.5):

    rescaled = base_weights * (1 - relevance_weight)
    rescaled["Relevance"] = relevance_weight
    return rescaled


results_with_relevance = {}

## Run TOPSIS with one more criterion - relevance score
for period_label, decision_df, base_weights, base_directions in [
    ("2030_2040", policy_evaluation_2030_2040, weights_p1, directions_p1),
    ("2040_2050", policy_evaluation_2040_2050, weights_p2, directions_p2),
]:
    for scen in alt_scenarios:
        combined_df = decision_df.copy()
        combined_df["Relevance"] = relevance_df[f"Relevance_{scen}"]

        combined_weights = add_relevance_weight(base_weights, relevance_weight=0.5)
        combined_directions = {**base_directions, "Relevance": "positive"}

        result = topsis(combined_df, combined_weights, combined_directions)
        key = f"{period_label}_{scen}"
        results_with_relevance[key] = result

        fname = f"TOPSIS_with_relevance_{key}.csv"
        result.to_csv(rf"D:/Tansy/Master thesis/MCDA/data/final/output/{fname}", index=True)    # <-- change this to your folder
        print(f"\n--- {key} (weights: {dict(combined_weights.round(4))}) ---")
        print(result.head())
        print(f"Saved: {fname}")

print(f"\n All 6 TOPSIS-with-relevance runs complete and saved.")


--- 2030_2040_MIX (weights: {'Cost_carbon_abatement': np.float64(0.0917), 'Social_acceptance_tech': np.float64(0.1542), 'Deployment_difficulty': np.float64(0.0917), 'Acceptance_policy_instrument': np.float64(0.025), 'Perceived_equity': np.float64(0.0917), 'Administrative_burden': np.float64(0.0458), 'Relevance': np.float64(0.5)}) ---
           Rank  TOPSIS Score  Distance to Ideal  Distance to Nadir  \
Policy_ID                                                             
I_01         13      0.470879           0.353871           0.314919   
N_02         31      0.463616           0.358939           0.310244   
I_03         46      0.430026           0.382028           0.288227   
I_04         13      0.470879           0.353871           0.314919   
I_05         31      0.463616           0.358939           0.310244   

           Cost_carbon_abatement Score  Social_acceptance_tech Score  \
Policy_ID                                                              
I_01                 

In [11]:
# Monte Carlo Uncertainty Analysis

## Set N_ITERATIONS - can be changed for a bigger/smaller run
N_ITERATIONS = 10000
RANDOM_SEED = 42  # for reproducibility - re-run with a different seed to sanity-check stability
np.random.seed(RANDOM_SEED)

## Monte Carlo draw of the Technology_policy_matrix

MATURE_TECHS = ["Biomass_Boiler", "Methane_Boiler", "CHP_Methane_heating",
                 "CCGT_Ren_Methane", "CHP_Methane_electricity"]
PV_EXCEPTION_PAIR = {"PV_Roof", "Alpine_PV"}

raw_tpm = data["tech_policy_matrix"].set_index("Policy_ID")
tech_cols_all = raw_tpm.columns.tolist()
tech_data_raw = data["technology"].set_index("Technology")

### Pre-identify which policies match the PV_Roof exception (support EXACTLY {PV_Roof, Alpine_PV}, nothing else)
pv_exception_policies = set()
for pid, row in raw_tpm.iterrows():
    supported = set(c for c in tech_cols_all if pd.notna(row[c]))
    if supported == PV_EXCEPTION_PAIR:
        pv_exception_policies.add(pid)
print(f"PV_Roof exception applies to policies: {sorted(pv_exception_policies)}")

### Function to perturb Technology_policy_matrix
def perturb_matrix_once():

    perturbed = raw_tpm.copy()

    for pid, row in raw_tpm.iterrows():
        supported = [c for c in tech_cols_all if pd.notna(row[c])]
        if len(supported) <= 1:
            continue  # single-tech policy: perturbation is a no-op after normalization

        if all(tech in MATURE_TECHS for tech in supported):
            continue  # all supported technologies are mature techs: no perturbation

        for tech in supported:
            if tech == "CHP_Methane_electricity":
                continue  # handled below, via the fixed ratio to heating
            current_val = row[tech]
            if tech in MATURE_TECHS:
                new_val = np.random.uniform(current_val, current_val * 1.2)
            elif tech == "PV_Roof" and pid in pv_exception_policies:
                new_val = np.random.uniform(current_val, current_val * 1.5)
            elif tech == "Light_EV":
                new_val = np.random.uniform(current_val, current_val * 1.5)
            else:
                new_val = current_val
            perturbed.loc[pid, tech] = new_val

        # CHP_Methane_electricity always follows the 46:44 ratio to heating
        if "CHP_Methane_electricity" in supported:
            if "CHP_Methane_heating" in supported:
                perturbed.loc[pid, "CHP_Methane_electricity"] = (
                    perturbed.loc[pid, "CHP_Methane_heating"] * (0.44 / 0.46)
                )
            else:
                current_val = row["CHP_Methane_electricity"]
                perturbed.loc[pid, "CHP_Methane_electricity"] = np.random.uniform(
                    current_val, current_val * 2
                )

    row_sums = perturbed[tech_cols_all].sum(axis=1, skipna=True)
    normalized = perturbed[tech_cols_all].div(row_sums, axis=0)
    return normalized

## Monte Carlo draw of Cost_current and Cost_2050
def perturb_costs_once():

    n = len(tech_data_raw)
    factor_current = np.random.uniform(0.8, 1.2, size=n)
    factor_2050 = np.random.uniform(0.8, 1.2, size=n)
    out = pd.DataFrame({
        "Technology": tech_data_raw.index,
        "Cost_current": tech_data_raw["Cost_current"].values * factor_current,
        "Cost_2050": tech_data_raw["Cost_2050"].values * factor_2050,
    })
    return out

## Monte Carlo draw of the 3 qualitative criteria tables
def perturb_qualitative_once():

    def sample_df(df):
        d = df.copy()
        d["sampled"] = [np.random.randint(lo, hi + 1) for lo, hi in zip(d["low"], d["high"])]
        return d
    return (
        sample_df(data["crit_acceptance_ins"]),
        sample_df(data["crit_admin_burden"]),
        sample_df(data["crit_perceived_equity"]),
    )

## Function for one run with the single Monte Carlo draw
def run_one_iteration():

    mc_weights = perturb_matrix_once()
    mc_costs = perturb_costs_once()
    mc_acc_ins, mc_admin, mc_equity = perturb_qualitative_once()

    mc_cost_2030_2040 = compute_weighted_cost(mc_weights, mc_costs, "Cost_current")
    mc_cost_2040_2050 = compute_weighted_cost(mc_weights, mc_costs, "Cost_2050")
    mc_cost_gap = mc_cost_2030_2040 - mc_cost_2040_2050

    mc_acceptance = compute_weighted_cost(mc_weights, data["technology"], "Acceptance")
    mc_deployment = compute_weighted_cost(mc_weights, data["scenario_production"], "EP2050_share")

    mc_acceptance_ins = lookup_by_category(data["policy"], "Instrument", mc_acc_ins, "Instrument", value_col="sampled")
    mc_admin_burden = lookup_by_category(data["policy"], "Administrative_touchpoint", mc_admin, "Administrative_touchpoint", value_col="sampled")
    policy_df_combo = data["policy"].copy()
    policy_df_combo["Instrument_Administrative_touchpoint"] = (
        policy_df_combo["Instrument"] + "_" + policy_df_combo["Administrative_touchpoint"]
    )
    mc_perceived_equity = lookup_by_category(policy_df_combo, "Instrument_Administrative_touchpoint",
                                              mc_equity, "Instrument_Administrative_touchpoint", value_col="sampled")

    mc_eval_p1 = pd.DataFrame({
        "Cost_carbon_abatement": mc_cost_2030_2040,
        "Social_acceptance_tech": mc_acceptance,
        "Deployment_difficulty": mc_deployment,
        "Acceptance_policy_instrument": mc_acceptance_ins,
        "Perceived_equity": mc_perceived_equity,
        "Administrative_burden": mc_admin_burden,
    })
    mc_eval_p2 = pd.DataFrame({
        "Cost_carbon_abatement": mc_cost_2040_2050,
        "Social_acceptance_tech": mc_acceptance,
        "Deployment_difficulty": mc_deployment,
        "Acceptance_policy_instrument": mc_acceptance_ins,
        "Perceived_equity": mc_perceived_equity,
        "Administrative_burden": mc_admin_burden,
        "Cost_gap": mc_cost_gap,
    })

    mc_topsis_p1 = topsis(mc_eval_p1, weights_p1, directions_p1)
    mc_topsis_p2 = topsis(mc_eval_p2, weights_p2, directions_p2)

    mc_relevance = pd.DataFrame(index=mc_weights.index)
    for scen in alt_scenarios:
        mc_relevance[f"Relevance_{scen}"] = compute_weighted_cost(
            mc_weights, tech_relevance_lookup, f"relevance_{scen}"
        )
    mc_relevance_groups = pd.DataFrame(index=mc_weights.index)
    for scen in alt_scenarios:
        labels_series, _ = ckmeans_classify(mc_relevance[f"Relevance_{scen}"], 5, group_labels)
        mc_relevance_groups[f"Group_{scen}"] = labels_series

    return mc_topsis_p1, mc_topsis_p2, mc_relevance, mc_relevance_groups


## Test run with 9 iterations first
print("\n--- test run: 3 quick iterations to check for errors before scaling up ---")
for i in range(3):
    p1, p2, rel, grp = run_one_iteration()
    print(f"Iteration {i}: Period1 top policy = {p1.index[p1["Rank"] == 1][0]}, "
          f"Period2 top policy = {p2.index[p2["Rank"] == 1][0]}, "
          f"Relevance_MIX range = [{rel['Relevance_MIX'].min():.3f}, {rel['Relevance_MIX'].max():.3f}]")
print(" Smoke test passed, no errors")

## Full Monte Carlo run
import time
print(f"\n--- Running full Monte Carlo: {N_ITERATIONS} iterations ---")
start_time = time.time()

topsis_p1_rows = []
topsis_p2_rows = []
relevance_rows = []

for it in range(N_ITERATIONS):
    p1, p2, rel, grp = run_one_iteration()

    p1_copy = p1[["Rank", "TOPSIS Score"]].copy()
    p1_copy["Iteration"] = it
    p1_copy["Policy_ID"] = p1_copy.index
    topsis_p1_rows.append(p1_copy)

    p2_copy = p2[["Rank", "TOPSIS Score"]].copy()
    p2_copy["Iteration"] = it
    p2_copy["Policy_ID"] = p2_copy.index
    topsis_p2_rows.append(p2_copy)

    for scen in alt_scenarios:
        rel_copy = pd.DataFrame({
            "Iteration": it,
            "Policy_ID": rel.index,
            "Scenario": scen,
            "Relevance": rel[f"Relevance_{scen}"].values,
            "Group": grp[f"Group_{scen}"].values,
        })
        relevance_rows.append(rel_copy)

    if (it + 1) % 100 == 0:
        elapsed = time.time() - start_time
        print(f"  {it + 1}/{N_ITERATIONS} done ({elapsed:.1f}s elapsed)")

elapsed_total = time.time() - start_time
print(f"\n Monte Carlo complete: {N_ITERATIONS} iterations in {elapsed_total:.1f}s "
      f"({elapsed_total / N_ITERATIONS * 10000:.1f}ms/iteration)")

mc_topsis_p1_full = pd.concat(topsis_p1_rows, ignore_index=True)[["Iteration", "Policy_ID", "Rank", "TOPSIS Score"]]
mc_topsis_p2_full = pd.concat(topsis_p2_rows, ignore_index=True)[["Iteration", "Policy_ID", "Rank", "TOPSIS Score"]]
mc_relevance_full = pd.concat(relevance_rows, ignore_index=True)

mc_topsis_p1_full.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/MC_TOPSIS_2030_2040.csv", index=False)    # <-- change this to your folder
mc_topsis_p2_full.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/MC_TOPSIS_2040_2050.csv", index=False)     # <-- change this to your folder
mc_relevance_full.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/MC_Relevance.csv", index=False)     # <-- change this to your folder

print(f"\n Saved: MC_TOPSIS_2030_2040.csv ({len(mc_topsis_p1_full)} rows)")
print(f" Saved: MC_TOPSIS_2040_2050.csv ({len(mc_topsis_p2_full)} rows)")
print(f" Saved: MC_Relevance.csv ({len(mc_relevance_full)} rows)")

## Stability check: how much does the TOP-ranked policy vary
print("\n--- Stability check: how often is each policy ranked #1 (Period 1)? ---")
print(mc_topsis_p1_full[mc_topsis_p1_full["Rank"] == 1]["Policy_ID"].value_counts().head(10))

PV_Roof exception applies to policies: ['I_16', 'I_17', 'I_18', 'I_19']

--- test run: 3 quick iterations to check for errors before scaling up ---
Iteration 0: Period1 top policy = I_36, Period2 top policy = N_07, Relevance_MIX range = [-0.458, 0.538]
Iteration 1: Period1 top policy = N_07, Period2 top policy = N_07, Relevance_MIX range = [-0.458, 0.538]
Iteration 2: Period1 top policy = N_07, Period2 top policy = N_07, Relevance_MIX range = [-0.458, 0.538]
 Smoke test passed, no errors

--- Running full Monte Carlo: 10000 iterations ---
  100/10000 done (1.9s elapsed)
  200/10000 done (3.8s elapsed)
  300/10000 done (5.7s elapsed)
  400/10000 done (7.9s elapsed)
  500/10000 done (9.9s elapsed)
  600/10000 done (11.8s elapsed)
  700/10000 done (13.7s elapsed)
  800/10000 done (15.5s elapsed)
  900/10000 done (17.7s elapsed)
  1000/10000 done (19.7s elapsed)
  1100/10000 done (21.6s elapsed)
  1200/10000 done (30.1s elapsed)
  1300/10000 done (33.1s elapsed)
  1400/10000 done (36.3s el

## 5 Result visualization
* Chart 1 — Relevance vs. TOPSIS Scatterplot (Period 1, 2030–2040)
  * Y-AXIS: TOPSIS Score
  * X-AXIS: Relevance Score
* Chart 2 - Combined table + dot-plot showing policy zones per scenario and TOPSIS rank shifts from Period 1
(2030-2040) to Period 2 (2040-2050)
* Chart 3 - AHP weight sensitivity analysis (Period 1 and Period 2)
  * Column chart of each TOPSIS criterion's AHP weight, with an error-bar-style whisker showing how far that  could move before the rank order changes
  * Two stacked subplots: Period 1 (6 criteria) and Period 2 (7 criteria, adds Cost_gap)
* Chart 4 - Relevance-compensated TOPSIS Top-10 policies
  * Period-1 policies per scenario, with Period-2 rank change shown as an arrow indicator inside each cell.
* Chart 5 - Monte Carlo uncertainty version of Chart 2
  * Same table layout (row sort, sector bar, name/status columns, dot plot spine)
  * in-cell horizontal bar showing the policy's MOST FREQUENT zone across 10000 MC iterations
  * a semi-transparent rounded "capsule" behind the P1/P2 dots, spanning the 25th-75th / 5th-95th percentile of the policy's Period-2 rank across 10000 MC iterations.

In [12]:
# Chart 1 — Relevance vs. TOPSIS Scatterplot (Period 1, 2030–2040)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

## resets all matplotlib settings to default
plt.rcdefaults()
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"

## Load & merge everything needed
policy_meta = data["policy"].set_index("Policy_ID")[["Status", "Energy_sector", "Promoting_technologies"]]
topsis_p1_full = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/TOPSIS_results_2030_2040.csv", index_col=0)
relevance_full = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/Policy_relevance_scores.csv", index_col=0)

plot_data = policy_meta.join(topsis_p1_full[["TOPSIS Score"]]).join(relevance_full)

## Color scheme
SECTOR_TECH_COLORS = {
    ("Electricity", "General"): "#6FA8D0",
    ("Electricity", "Hydro Power"): "#3E7CB1",
    ("Electricity", "PV & Wind"): "#0F6B99",
    ("Electricity", "CHP & CCGT"): "#1B3B5F",
    ("Heating", "General"): "#F2A65A",
    ("Heating", "HP"): "#D2691E",
    ("Heating", "Boiler"): "#A0451A",
    ("Transport", "General"): "#C39BD3",
    ("Transport", "FCEV"): "#9B59B6",
    ("Transport", "EV"): "#6C3483",
}
STATUS_MARKER = {"I": "o", "N": "^"}
STATUS_LABEL = {"I": "Existing policy", "N": "Fictitious policy"}

ZONE_COLORS = {
    "Strong negative": ("#595959", "Scale back", "white"),
    "Weak negative": ("#A6A6A6", "Modest scale back", "#404040"),
    "Neutral_low": ("#E8DCC0", "Maintain — review", "#6B5A2A"),
    "Neutral_high": ("#C9AD7D", "Maintain — confirmed", "#4A3D1C"),
    "Weak positive_low": ("#D3E8C8", "Modest expansion — review", "#355C2A"),
    "Weak positive_high": ("#8FCB84", "Modest expansion — confirmed", "#1F4A17"),
    "Strong positive_low": ("#5FAE55", "Strong expansion — review", "#153D10"),
    "Strong positive_high": ("#2E7D32", "Strong expansion — confirmed", "white"),
}
TOPSIS_SPLIT = 0.6
GROUP_ORDER = ["Strong negative", "Weak negative", "Neutral", "Weak positive", "Strong positive"]
SYMLOG_LINTHRESH = 0.02  # region around 0 treated as linear (avoids log(0)); tuned to this data's cluster width

## Global Y-axis range (identical on every plot)
y_min = plot_data["TOPSIS Score"].min()
y_max = plot_data["TOPSIS Score"].max()
y_pad = (y_max - y_min) * 0.08
GLOBAL_YLIM = (y_min - y_pad, y_max + y_pad)
print(f"Global Y-axis (TOPSIS Score) range, fixed on every plot: {GLOBAL_YLIM}")

## Draw the 8 background zones

### Computes VISUAL boundary positions for the background zones
def get_zone_breaks(scenario):
    """Re-derives the exact Ckmeans.1d.dp breakpoints (with the zero-in-Neutral
    guarantee) for a scenario's relevance scores, matching Step 8 exactly."""
    labels_series, breaks = ckmeans_classify(relevance_full[f"Relevance_{scenario}"], 5, GROUP_ORDER)
    return breaks  # [min, b1, b2, b3, b4, max]

### Find the MIDPOINT between the last point of one group and the first point of the next
def get_visual_zone_edges(values, group_labels, breaks):

    edges = [breaks[0]]
    for i in range(len(GROUP_ORDER) - 1):
        this_group_vals = values[group_labels == GROUP_ORDER[i]]
        next_group_vals = values[group_labels == GROUP_ORDER[i + 1]]
        if len(this_group_vals) > 0 and len(next_group_vals) > 0:
            midpoint = (this_group_vals.max() + next_group_vals.min()) / 2
        else:
            midpoint = breaks[i + 1]  # fallback if a group is empty
        edges.append(midpoint)
    edges.append(breaks[-1])
    return edges

### Draw the 8 background zones
def draw_background_zones(ax, x_edges_raw, xlim, ylim):

    x_edges = [xlim[0]] + list(x_edges_raw[1:5]) + [xlim[1]]

    def label_zone(x_left, x_right, y_bottom, y_top, hex_color):
        ax.add_patch(mpatches.Rectangle(
            (x_left, y_bottom), x_right - x_left, y_top - y_bottom,
            facecolor=hex_color, edgecolor="none", zorder=0))

    # Strong negative / Weak negative: single band, full height
    for i, group in enumerate(["Strong negative", "Weak negative"]):
        hex_color, label, textcolor = ZONE_COLORS[group]
        label_zone(x_edges[i], x_edges[i + 1], ylim[0], ylim[1], hex_color)

    # Neutral / Weak positive / Strong positive: split at TOPSIS=0.6
    for i, group in zip([2, 3, 4], ["Neutral", "Weak positive", "Strong positive"]):
        hex_low, label_low, text_low = ZONE_COLORS[f"{group}_low"]
        hex_high, label_high, text_high = ZONE_COLORS[f"{group}_high"]
        label_zone(x_edges[i], x_edges[i + 1], ylim[0], TOPSIS_SPLIT, hex_low)
        label_zone(x_edges[i], x_edges[i + 1], TOPSIS_SPLIT, ylim[1], hex_high)

## Scatters policy points with sector/tech color, status shape, white outline.
def draw_points(ax, df, point_size=70):
    for (sector, tech), marker_status_group in df.groupby(["Energy_sector", "Promoting_technologies"]):
        color = SECTOR_TECH_COLORS[(sector, tech)]
        for status, sub in marker_status_group.groupby("Status"):
            ax.scatter(sub["Relevance"], sub["TOPSIS Score"],
                       c=color, marker=STATUS_MARKER[status], s=point_size,
                       edgecolors="white", linewidths=1.5, zorder=3)

## Auto-zooms x-axis to this panel's own data range, with padding with using a SYMMETRIC LOG scale.
def set_xlim_auto(ax, df, pad_frac=0.08):

    ax.set_xscale("symlog", linthresh=SYMLOG_LINTHRESH, linscale=1.0)
    x_min, x_max = df["Relevance"].min(), df["Relevance"].max()
    x_range = x_max - x_min
    if x_range == 0:
        x_range = abs(x_max) if x_max != 0 else 1.0
    pad = x_range * pad_frac
    ax.set_xlim(x_min - pad, x_max + pad)

    default_ticks = list(ax.get_xticks())
    extra_ticks = [t for t in [x_min, x_max] if abs(t) > SYMLOG_LINTHRESH]  # only if outside the linear zone
    all_ticks = sorted(set([t for t in default_ticks if x_min - pad <= t <= x_max + pad] + extra_ticks))
    ax.set_xticks(all_ticks)


def build_zone_legend_handles():

    handles = []
    for key, (hex_color, label, _) in ZONE_COLORS.items():
        handles.append(mpatches.Patch(facecolor=hex_color, edgecolor="none", label=label))
    return handles

## Draws the 8 background zone colors + labels, placed BELOW the plot.
def add_zone_legend(ax_or_fig, x0, y0, width, is_fig=False, fontsize=10.5, row_gap=0.03):

    from matplotlib.legend import Legend

    items = list(ZONE_COLORS.items())
    row1_items = items[:4]
    row2_items = items[4:]

    target = ax_or_fig
    transform = target.transFigure if is_fig else target.transAxes
    y = y0
    for row_items in [row1_items, row2_items]:
        handles = [mpatches.Patch(facecolor=hex_color, edgecolor="none", label=label)
                   for _, (hex_color, label, _) in row_items]
        leg = Legend(target, handles, [h.get_label() for h in handles],
                     loc="upper left", bbox_to_anchor=(x0, y), bbox_transform=transform,
                     ncol=len(handles), fontsize=fontsize, frameon=False,
                     handletextpad=0.35, columnspacing=1.3, borderpad=0.1, borderaxespad=0)
        target.add_artist(leg)
        y -= row_gap

## Draws a 4-ROW legend (Electricity techs / Heating techs / Transport techs/ Status) ABOVE the plot
def add_point_legend(ax_or_fig, x0, y0, width, is_fig=False, fontsize=10.5, row_gap=0.03):

    from matplotlib.legend import Legend

    rows = []
    for sector in ["Electricity", "Heating", "Transport"]:
        row = [(f"{sector} — {tech}", color) for (s, tech), color in SECTOR_TECH_COLORS.items() if s == sector]
        rows.append(row)
    status_row = [(STATUS_LABEL[s], "grey", STATUS_MARKER[s]) for s in STATUS_MARKER]

    y = y0
    target = ax_or_fig
    transform = target.transFigure if is_fig else target.transAxes
    for row in rows:
        handles = [Line2D([0], [0], marker="o", color="none", markerfacecolor=c,
                           markeredgecolor="white", markersize=10, label=lbl) for lbl, c in row]
        leg = Legend(target, handles, [h.get_label() for h in handles],
                     loc="upper left", bbox_to_anchor=(x0, y), bbox_transform=transform,
                     ncol=len(handles), fontsize=fontsize, frameon=False,
                     handletextpad=0.25, columnspacing=1.1, borderpad=0.1, borderaxespad=0)
        target.add_artist(leg)
        y -= row_gap

    handles = [Line2D([0], [0], marker=m, color="none", markerfacecolor="grey",
                       markeredgecolor="white", markersize=10, label=lbl) for lbl, _, m in status_row]
    leg = Legend(target, handles, [h.get_label() for h in handles],
                 loc="upper left", bbox_to_anchor=(x0, y), bbox_transform=transform,
                 ncol=len(handles), fontsize=fontsize, frameon=False,
                 handletextpad=0.25, columnspacing=1.1, borderpad=0.1, borderaxespad=0)
    target.add_artist(leg)


def style_axis(ax, scenario):
    ax.set_ylim(GLOBAL_YLIM)
    ax.set_xlabel(f"Relevance score ({scenario})", fontsize=10)
    ax.set_ylabel("TOPSIS Score", fontsize=10)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2g}"))

## Build the 6 figures: for each scenario, (A) standalone all-sector plot, (B) 2x2 combined figure (all-sector + 3 sector panels)

sectors = ["Electricity", "Heating", "Transport"]

for scenario in ["MIX", "REMIX", "H2"]:
    scen_df = plot_data.copy()
    scen_df["Relevance"] = plot_data[f"Relevance_{scenario}"]
    labels_series, breaks = ckmeans_classify(relevance_full[f"Relevance_{scenario}"], 5, GROUP_ORDER)
    visual_edges = get_visual_zone_edges(relevance_full[f"Relevance_{scenario}"], labels_series, breaks)

    # (A) Standalone all-sector figure
    fig = plt.figure(figsize=(11, 9.5))
    # Fixed axes rect: [left, bottom, width, height] in figure fraction -
    # top ~32% reserved for title + 4-row point legend (above plot),
    # bottom ~18% reserved for the 2-row zone-color legend (below plot).
    ax_rect = [0.08, 0.24, 0.88, 0.55]
    ax = fig.add_axes(ax_rect)
    set_xlim_auto(ax, scen_df)
    style_axis(ax, scenario)
    draw_background_zones(ax, visual_edges, ax.get_xlim(), GLOBAL_YLIM)
    draw_points(ax, scen_df, point_size=70)

    fig.text(0.5, 0.97, f"Relevance vs TOPSIS Score — {scenario} (2030–2040), All Sectors",
              fontsize=13, fontweight="bold", ha="center", va="top")
    legend_top = ax_rect[1] + ax_rect[3] + 0.13  # band between title and axes top
    add_point_legend(fig, x0=ax_rect[0], y0=legend_top, width=ax_rect[2], is_fig=True,
                      fontsize=9.5, row_gap=0.03)
    add_zone_legend(fig, x0=ax_rect[0], y0=ax_rect[1] - 0.08, width=ax_rect[2], is_fig=True,
                     fontsize=9.5, row_gap=0.03)

    fname_a = f"Chart1_A_{scenario.replace(' ', '')}_2030_2040.png"
    fig.savefig(fr"D:/Tansy/Master thesis/MCDA/data/final/output/{fname_a}", dpi=150, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"Saved: {fname_a}")

    # (B) 2x2 combined figure
    fig = plt.figure(figsize=(11, 9.5))
    grid_top = 0.75  # bottom of the reserved title+legend band, subplots start here
    grid_bottom = 0.18  # top of the reserved zone-legend band, subplots end here
    gs = fig.add_gridspec(2, 2, left=0.06, right=0.98, bottom=grid_bottom, top=grid_top,
                           wspace=0.28, hspace=0.38)

    panels = [("All Sectors", scen_df)] + [(s, scen_df[scen_df["Energy_sector"] == s]) for s in sectors]
    panel_letters = ["(a)", "(b)", "(c)", "(d)"]

    axes_list = []
    for idx, (panel_name, panel_df) in enumerate(panels):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        set_xlim_auto(ax, scen_df)
        style_axis(ax, scenario)
        draw_background_zones(ax, visual_edges, ax.get_xlim(), GLOBAL_YLIM)
        draw_points(ax, panel_df, point_size=35)
        ax.set_title(f"{panel_letters[idx]}  {panel_name}", fontsize=12, fontweight="bold", loc="left")
        axes_list.append(ax)

    fig.text(0.5, 0.97, f"Relevance vs TOPSIS Score — {scenario} (2030–2040)",
              fontsize=13, fontweight="bold", ha="center", va="top")
    legend_top_b = grid_top + 0.16
    add_point_legend(fig, x0=0.08, y0=legend_top_b, width=0.88, is_fig=True,
                      fontsize=9.5, row_gap=0.03)
    add_zone_legend(fig, x0=0.08, y0=grid_bottom - 0.08, width=0.88, is_fig=True,
                     fontsize=9.5, row_gap=0.03)

    fname_b = f"Chart1_B_{scenario.replace(' ', '')}_2030_2040_grid.png"
    fig.savefig(fr"D:/Tansy/Master thesis/MCDA/data/final/output/{fname_b}", dpi=150, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"Saved: {fname_b}")

print("\n All 6 Chart Set 1 figures generated.")

Global Y-axis (TOPSIS Score) range, fixed on every plot: (np.float64(0.41828856791921903), np.float64(0.8701003069881832))
Saved: Chart1_A_MIX_2030_2040.png
Saved: Chart1_B_MIX_2030_2040_grid.png
Saved: Chart1_A_REMIX_2030_2040.png
Saved: Chart1_B_REMIX_2030_2040_grid.png
Saved: Chart1_A_H2_2030_2040.png
Saved: Chart1_B_H2_2030_2040_grid.png

 All 6 Chart Set 1 figures generated.


In [13]:
# Chart 2 Combined table + dot-plot

from matplotlib.legend import Legend
import textwrap

## Assemble the merged dataset
policy_full = data["policy"].set_index("Policy_ID")[["Name", "Status", "Energy_sector", "Promoting_technologies"]]
t1 = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/TOPSIS_results_2030_2040.csv", index_col=0)[["Rank", "TOPSIS Score"]].rename(
    columns={"Rank": "Rank_P1", "TOPSIS Score": "TOPSIS_P1"})
t2 = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/TOPSIS_results_2040_2050.csv", index_col=0)[["Rank"]].rename(columns={"Rank": "Rank_P2"})
relevance = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/Policy_relevance_scores.csv", index_col=0)

chart2_df = policy_full.join(t1).join(t2).join(relevance)

## Classify policy zones based on the Chart set 1
def classify_zone(group, topsis_score):
    if group in ("Strong negative", "Weak negative"):
        return group
    suffix = "_high" if topsis_score >= TOPSIS_SPLIT else "_low"
    return f"{group}{suffix}"


for scen in ["MIX", "REMIX", "H2"]:
    chart2_df[f"Zone_{scen}"] = [
        classify_zone(g, t) for g, t in zip(chart2_df[f"Group_{scen}"], chart2_df["TOPSIS_P1"])
    ]

SECTOR_ORDER = ["Electricity", "Heating", "Transport"]
TECH_ORDER = {
    "Electricity": ["General", "Hydro Power", "PV & Wind", "CHP & CCGT"],
    "Heating": ["General", "HP", "Boiler"],
    "Transport": ["General", "FCEV", "EV"],
}
chart2_df["sector_rank"] = chart2_df["Energy_sector"].map({s: i for i, s in enumerate(SECTOR_ORDER)})
chart2_df["tech_rank"] = chart2_df.apply(
    lambda r: TECH_ORDER[r["Energy_sector"]].index(r["Promoting_technologies"]), axis=1)
chart2_df["status_rank"] = chart2_df["Status"].map({"I": 0, "N": 1})
chart2_df = chart2_df.sort_values(["sector_rank", "tech_rank", "status_rank"]).reset_index()

n_rows = len(chart2_df)
SECTOR_BAR_COLORS = {"Electricity": "#3E7CB1", "Heating": "#D2691E", "Transport": "#9B59B6"}
STATUS_TEXT = {"I": "Existing", "N": "Fictitious"}
P1_COLOR = "#2C3E50"
P2_COLOR = "#E8A33D"
IMPROVE_COLOR = "#2E7D32"
WORSEN_COLOR = "#C0392B"

## Wrap names, compute a UNIFORM row height (fits up to 2 lines)
WRAP_WIDTH = 80  # characters per line - wide enough that almost all names fit on 1 line
LINE_H = 0.155     # inches per text line
INTER_GAP = 0.06   # extra inches between policies (added on top of the row height)

chart2_df["lines"] = chart2_df["Name"].apply(lambda n: textwrap.wrap(n, WRAP_WIDTH) or [n])
chart2_df["n_lines"] = chart2_df["lines"].apply(len)
max_lines = chart2_df["n_lines"].max()
print(f"Max lines needed at wrap={WRAP_WIDTH}: {max_lines}, "
      f"policies needing >1 line: {(chart2_df['n_lines'] > 1).sum()}")

ROW_H = max_lines * LINE_H  # SAME height for every row, regardless of that row's own line count
chart2_df["y_top"] = chart2_df.index * (ROW_H + INTER_GAP)
chart2_df["y_bottom"] = chart2_df["y_top"] + ROW_H
chart2_df["y_center"] = (chart2_df["y_top"] + chart2_df["y_bottom"]) / 2
table_height_in = chart2_df["y_bottom"].iloc[-1]

print(f"Total policies: {n_rows}, table height: {table_height_in:.1f}in, "
      f"policies needing wrap (2+ lines): {(chart2_df['n_lines'] > 1).sum()}")


def truncate_none(name):
    return name  # no truncation anymore - full wrapped text is shown

## Build the figure: sector bar | name | status | 3 category cells | dot plot

fig_height = table_height_in + 2.6
fig = plt.figure(figsize=(16, fig_height))

width_ratios = [0.14, 3.3, 0.5, 1.6, 2.8]
top_margin_in = 1.35
bottom_margin_in = 1.3
gs = fig.add_gridspec(1, 5, width_ratios=width_ratios,
                       left=0.03, right=0.99,
                       bottom=bottom_margin_in / fig_height,
                       top=1 - top_margin_in / fig_height,
                       wspace=0.0)

ax_sector = fig.add_subplot(gs[0, 0])
ax_name = fig.add_subplot(gs[0, 1], sharey=ax_sector)
ax_status = fig.add_subplot(gs[0, 2], sharey=ax_sector)
ax_cat = fig.add_subplot(gs[0, 3], sharey=ax_sector)
ax_dot = fig.add_subplot(gs[0, 4], sharey=ax_sector)

for ax in [ax_sector, ax_name, ax_status, ax_cat, ax_dot]:
    ax.set_ylim(table_height_in, -0.35)  # row 0 (data top) at figure top
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

### Sector indicator bar
ax_sector.set_xlim(0, 1)
for _, r in chart2_df.iterrows():
    ax_sector.add_patch(mpatches.Rectangle((0, r["y_top"]), 1, r["y_bottom"] - r["y_top"],
                         facecolor=SECTOR_BAR_COLORS[r["Energy_sector"]], edgecolor="none"))

### Policy name column (wrapped, vertically centered within fixed row height)
ax_name.set_xlim(0, 1)
for _, r in chart2_df.iterrows():
    n = len(r["lines"])
    # Center the block of n lines within the row's fixed height
    start_y = r["y_center"] - (n - 1) * LINE_H / 2
    for i, line in enumerate(r["lines"]):
        ax_name.text(0.008, start_y + i * LINE_H, line, va="center", ha="left", fontsize=10)

### Status column
ax_status.set_xlim(0, 1)
for _, r in chart2_df.iterrows():
    ax_status.text(0.03, r["y_center"], STATUS_TEXT[r["Status"]], va="center", ha="left", fontsize=10)

### category color cells (MIX, REMIX, H2)
ax_cat.set_xlim(0, 3)
for _, r in chart2_df.iterrows():
    for i, scen in enumerate(["MIX", "REMIX", "H2"]):
        hex_color, _, _ = ZONE_COLORS[r[f"Zone_{scen}"]]
        ax_cat.add_patch(mpatches.Rectangle((i + 0.03, r["y_top"]), 0.94, r["y_bottom"] - r["y_top"],
                          facecolor=hex_color, edgecolor="white", linewidth=0.5))
ax_cat.set_xticks([0.5, 1.5, 2.5])
ax_cat.xaxis.set_ticks_position("top")
ax_cat.set_xticklabels(["MIX", "RE MIX", "H2"], fontsize=10, fontweight="bold")
ax_cat.tick_params(axis="x", length=0, pad= -18)

### Rank change dot plot
changes = (chart2_df["Rank_P2"] - chart2_df["Rank_P1"])
max_abs_change = changes.abs().max()
MAX_OFFSET = 3.2
scale = MAX_OFFSET / max_abs_change if max_abs_change > 0 else 1

SPINE_X = 0
ax_dot.set_xlim(-MAX_OFFSET - 2.0, MAX_OFFSET + 0.6)

for _, r in chart2_df.iterrows():
    y = r["y_center"]
    change = int(r["Rank_P2"] - r["Rank_P1"])
    ax_dot.scatter([SPINE_X], [y], s=250, c=P1_COLOR, edgecolors="white", linewidths=1.2, zorder=3)
    ax_dot.text(SPINE_X, y, str(int(r["Rank_P1"])), color="white", fontsize=9,
                ha="center", va="center", fontweight="bold", zorder=4)

    if change != 0:
        dx = abs(change) * scale if change < 0 else -abs(change) * scale
        line_color = IMPROVE_COLOR if change < 0 else WORSEN_COLOR
        ax_dot.plot([SPINE_X, dx], [y, y], color=line_color, linewidth=1.6, zorder=2)
        ax_dot.scatter([dx], [y], s=250, c=P2_COLOR, edgecolors="white", linewidths=1.2, zorder=3)
        label = f"{change:+d}"
        label_x = dx + (0.35 if dx > 0 else -0.35)
        ax_dot.text(label_x, y, label, color=line_color, fontsize=9, fontweight="bold",
                    ha="left" if dx > 0 else "right", va="center", zorder=4)

### Column headers
fig.canvas.draw()
pos_name = ax_name.get_position()
pos_status = ax_status.get_position()
pos_cat = ax_cat.get_position()
pos_dot = ax_dot.get_position()

header_y = pos_name.y1 + 0.002
fig.text(pos_name.x0 + 0.18, header_y, "Policy", fontsize=12, fontweight="bold", va="bottom", ha="center")
fig.text(pos_status.x0 + 0.018, header_y, "Status", fontsize=12, fontweight="bold", va="bottom", ha="center")
fig.text((pos_cat.x0 + pos_cat.x1) / 2, header_y, "Scenario", fontsize=12, fontweight="bold", va="bottom", ha="center")
fig.text((pos_dot.x0 + pos_dot.x1) / 2, header_y, "TOPSIS Rank 2030-2040 → 2040-2050",
          fontsize=12, fontweight="bold", va="bottom", ha="center")

## Title
fig.text(0.5, 0.995, "Policy Category Overview & TOPSIS Rank Shift (2030–2040 → 2040–2050)",
          fontsize=15, fontweight="bold", ha="center", va="top")

## Top legend: sector colors (left) + dot colors, positioned directly ABOVE

sector_handles = [Line2D([0], [0], marker="s", color="none", markerfacecolor=c, markeredgecolor="none",
                          markersize=11, label=s) for s, c in SECTOR_BAR_COLORS.items()]
dot_handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor=P1_COLOR, markeredgecolor="white",
           markersize=11, label="2030-2040 TOPSIS Rank"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor=P2_COLOR, markeredgecolor="white",
           markersize=11, label="2040-2050 TOPSIS Rank"),
]
legend_y = 0.975
leg1 = Legend(fig, sector_handles, [h.get_label() for h in sector_handles],
              loc="upper left", bbox_to_anchor=(0.03, legend_y), bbox_transform=fig.transFigure,
              ncol=3, fontsize=10, frameon=False)
fig.add_artist(leg1)
dot_legend_x = (pos_dot.x0 + pos_dot.x1) / 2
leg2 = Legend(fig, dot_handles, [h.get_label() for h in dot_handles],
              loc="upper center", bbox_to_anchor=(dot_legend_x, legend_y), bbox_transform=fig.transFigure,
              ncol=2, fontsize=10, frameon=False)
fig.add_artist(leg2)

## Bottom legend: 8 zone colors (same style as Chart 1)
add_zone_legend(fig, x0=0.03, y0=bottom_margin_in / fig_height - 0.35 / fig_height,
                 width=0.94, is_fig=True, fontsize=10, row_gap=0.012)

fname = "Chart2_policy_table_rank_change.png"
fig.savefig(fr"D:/Tansy/Master thesis/MCDA/data/final/output/{fname}", dpi=150, bbox_inches="tight", pad_inches=0.3)
plt.close(fig)
print(f"\n Saved: {fname}")

Max lines needed at wrap=80: 2, policies needing >1 line: 3
Total policies: 48, table height: 17.7in, policies needing wrap (2+ lines): 3

 Saved: Chart2_policy_table_rank_change.png


In [14]:
# Chart 3 AHP weight sensitivity analysis (Period 1 and Period 2)

BAR_COLOR = "#2E6F8E"
WHISKER_COLOR = "#E8A33D"

# Original defined order
ORDER_P1 = ["Cost_carbon_abatement", "Social_acceptance_tech", "Deployment_difficulty",
            "Acceptance_policy_instrument", "Perceived_equity", "Administrative_burden"]
ORDER_P2 = ORDER_P1 + ["Cost_gap"]

LABEL_MAP = {
    "Cost_carbon_abatement": "Cost of carbon\nabatement",
    "Social_acceptance_tech": "Acceptance of\ntechnologies",
    "Deployment_difficulty": "Deployment\ndifficulty",
    "Acceptance_policy_instrument": "Acceptance of\npolicy instrument",
    "Perceived_equity": "Perceived\nequity",
    "Administrative_burden": "Administrative\nburden",
    "Cost_gap": "Cost gap",
}


def plot_period(ax, period_label, order, panel_letter, title):
    sub = sensitivity_results[sensitivity_results["Period"] == period_label].set_index("Criterion").loc[order]
    x = range(len(order))
    weights = sub["Current_Weight"].values
    lower_err = sub["Allowed_Decrease"].values
    upper_err = sub["Allowed_Increase"].values

    ax.bar(x, weights, color=BAR_COLOR, width=0.6, zorder=2)
    ax.errorbar(x, weights, yerr=[lower_err, upper_err], fmt="none",
                ecolor=WHISKER_COLOR, elinewidth=2, capsize=6, capthick=2, zorder=3)

    ax.set_xticks(list(x))
    ax.set_xticklabels([LABEL_MAP[c] for c in order], fontsize=8.5)
    ax.set_ylabel("BWM weight", fontsize=9.5)
    ax.tick_params(axis="y", labelsize=8.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(f"{panel_letter}  {title}", fontsize=11, fontweight="bold", loc="left", pad=10)
    ax.margins(y=0.15)  # a bit of headroom above the tallest whisker


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 9.5))
plot_period(ax1, "2030_2040", ORDER_P1, "(a)", "Period 1 (2030–2040) — 6 criteria")
plot_period(ax2, "2040_2050", ORDER_P2, "(b)", "Period 2 (2040–2050) — 7 criteria")

fig.suptitle("Criterion Weight Sensitivity — 2030–2040 vs. 2040–2050",
             fontsize=13, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96])

fname = "Chart3_weight_sensitivity.png"
fig.savefig(fr"D:/Tansy/Master thesis/MCDA/data/final/output/{fname}", dpi=150, bbox_inches="tight", pad_inches=0.3)
plt.close(fig)
print(f"\n Saved: {fname}")


 Saved: Chart3_weight_sensitivity.png


In [15]:
# Chart 4 - Relevance-compensated TOPSIS Top-10 policies

policy_names = data["policy"].set_index("Policy_ID")["Name"]
relevance = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/Policy_relevance_scores.csv", index_col=0)

SCENARIOS = ["MIX", "REMIX", "H2"]

def classify_zone(group, topsis_score):
    if group in ("Strong negative", "Weak negative"):
        return group
    suffix = "_high" if topsis_score >= TOPSIS_SPLIT else "_low"
    return f"{group}{suffix}"

scenario_tables = {}
for scen in SCENARIOS:
    p1 = pd.read_csv(fr"D:/Tansy/Master thesis/MCDA/data/final/output/TOPSIS_with_relevance_2030_2040_{scen}.csv", index_col=0)
    p2 = pd.read_csv(fr"D:/Tansy/Master thesis/MCDA/data/final/output/TOPSIS_with_relevance_2040_2050_{scen}.csv", index_col=0)

    top10 = p1.sort_values("Rank").head(10).copy()
    top10["Rank_P2"] = p2.loc[top10.index, "Rank"]
    top10["Change"] = top10["Rank_P2"] - top10["Rank"]
    top10["Name"] = policy_names.loc[top10.index]
    top10["Zone"] = [
        classify_zone(relevance.loc[pid, f"Group_{scen}"], top10.loc[pid, "TOPSIS Score"])
        for pid in top10.index
    ]
    scenario_tables[scen] = top10.reset_index()
    print(f"\n{scen} top 10:")
    print(top10[["Name", "Rank", "Rank_P2", "Change", "Zone"]].to_string())

IMPROVE_COLOR = "#2E7D32"
WORSEN_COLOR = "#C0392B"
NEUTRAL_COLOR = "#7F7F7F"

def wrap_name(name, width=55):
    lines = textwrap.wrap(name, width)
    return lines[:2] if len(lines) > 2 else lines  # cap at 2 lines, matching Chart 2's approach

# Build the figure

N_ROWS = 10
ROW_H = 0.72  # inches (a bit taller, to comfortably fit 2-line wrapped names)
fig_height = N_ROWS * ROW_H + 3.0
fig = plt.figure(figsize=(16, fig_height))

width_ratios = [0.9, 4.6, 4.6, 4.6]
top_margin_in = 1.5
bottom_margin_in = 1.4
gs = fig.add_gridspec(1, 4, width_ratios=width_ratios,
                       left=0.03, right=0.98,
                       bottom=bottom_margin_in / fig_height,
                       top=1 - top_margin_in / fig_height,
                       wspace=0.03)

ax_rankcol = fig.add_subplot(gs[0, 0])
ax_scen = [fig.add_subplot(gs[0, i + 1], sharey=ax_rankcol) for i in range(3)]

for ax in [ax_rankcol] + ax_scen:
    ax.set_ylim(N_ROWS, 0)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

## Add "TOPSIS Rank" column
ax_rankcol.set_xlim(0, 1)
for i in range(N_ROWS):
    ax_rankcol.text(0.5, i + 0.5, str(i + 1), ha="center", va="center",
                     fontsize=13, fontweight="bold")

## Add Scenario columns
for ax, scen in zip(ax_scen, SCENARIOS):
    ax.set_xlim(0, 1)
    df = scenario_tables[scen]
    for i, row in df.iterrows():
        hex_color, _, textcolor = ZONE_COLORS[row["Zone"]]
        ax.add_patch(mpatches.Rectangle((0.02, i + 0.06), 0.96, 0.88,
                     facecolor=hex_color, edgecolor="white", linewidth=1))
        lines = wrap_name(row["Name"])
        n = len(lines)
        line_h_frac = 0.19
        start_y = (i + 0.5) - (n - 1) * line_h_frac / 2
        for li, line in enumerate(lines):
            ax.text(0.06, start_y + li * line_h_frac, line, ha="left", va="center",
                    fontsize=9, color=textcolor)

        change = int(row["Change"])
        if change < 0:
            arrow_color, arrow, label = IMPROVE_COLOR, "▲", f"{abs(change)}"
        elif change > 0:
            arrow_color, arrow, label = WORSEN_COLOR, "▼", f"{abs(change)}"
        else:
            arrow_color, arrow, label = NEUTRAL_COLOR, "–", ""
        badge_text = f"{arrow} {label}".strip()
        ax.text(0.94, i + 0.5, badge_text, ha="right", va="center", fontsize=10,
                fontweight="bold", color=arrow_color,
                bbox=dict(boxstyle="round,pad=0.28", facecolor="white",
                          alpha=0.72, edgecolor="none"))

## Add Column headers
fig.canvas.draw()
pos_rank = ax_rankcol.get_position()
header_y = pos_rank.y1 + 0.01
fig.text((pos_rank.x0 + pos_rank.x1) / 2, header_y, "TOPSIS\nRank",
          fontsize=10.5, fontweight="bold", ha="center", va="bottom")
for ax, scen in zip(ax_scen, SCENARIOS):
    pos = ax.get_position()
    fig.text((pos.x0 + pos.x1) / 2, header_y, scen, fontsize=12, fontweight="bold",
              ha="center", va="bottom")

## Add Title
fig.text(0.5, 0.995,
          "Relevance-Compensated TOPSIS Top-10 Policies per Scenario (2030-2040)\nwith 2040-2050 Rank Change",
          fontsize=15, fontweight="bold", ha="center", va="top")

## Arrow-color legend, above the table
arrow_handles = [
    Line2D([0], [0], marker="^", color="none", markerfacecolor=IMPROVE_COLOR, markersize=10,
           label="Rank improved in Period 2"),
    Line2D([0], [0], marker="v", color="none", markerfacecolor=WORSEN_COLOR, markersize=10,
           label="Rank worsened in Period 2"),
    Line2D([0], [0], marker="_", color="none", markeredgecolor=NEUTRAL_COLOR, markersize=10, markeredgewidth=3,
           label="Rank unchanged"),
]
legend_y = header_y + 0.08
leg1 = Legend(fig, arrow_handles, [h.get_label() for h in arrow_handles],
              loc="upper center", bbox_to_anchor=(0.5, legend_y), bbox_transform=fig.transFigure,
              ncol=3, fontsize=10, frameon=False)
fig.add_artist(leg1)

## Bottom legend: 8 zone colors (same style as earlier charts)
add_zone_legend(fig, x0=0.03, y0=bottom_margin_in / fig_height - 0.35 / fig_height,
                 width=0.94, is_fig=True, fontsize=10, row_gap=0.03)

fname = "Chart4_top10_per_scenario.png"
fig.savefig(fr"D:/Tansy/Master thesis/MCDA/data/final/output/{fname}", dpi=150, bbox_inches="tight", pad_inches=0.3)
plt.close(fig)
print(f"\n Saved: {fname}")


MIX top 10:
                                                              Name  Rank  Rank_P2  Change                  Zone
Policy_ID                                                                                                      
I_12                                         Subsidy on heat pumps     1        1       0  Strong positive_high
N_07       Carbon removal for fossil electricity generation plants     2        2       0    Weak positive_high
N_08                                          P+D subsidy on DACCS     3        3       0    Weak positive_high
I_29                 Investment contribution (IB)  for wind energy     5        5       0           Neutral_low
I_30                       Project planning grants for wind energy     5        5       0           Neutral_low
I_28                  Sliding market premium (GMP) for wind energy     5        5       0           Neutral_low
N_09           Subsidy on CO2 transport and storage infrastructure     7        7       0  

In [16]:
# Chart 5 Monte Carlo uncertainty version of Chart 2

from matplotlib.patches import FancyBboxPatch

## Assemble the base table (same as Chart 2)
policy_full = data["policy"].set_index("Policy_ID")[["Name", "Status", "Energy_sector", "Promoting_technologies"]]
t1 = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/TOPSIS_results_2030_2040.csv", index_col=0)[["Rank", "TOPSIS Score"]].rename(columns={"Rank": "Rank_P1", "TOPSIS Score": "TOPSIS_P1"})
t2 = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/TOPSIS_results_2040_2050.csv", index_col=0)[["Rank"]].rename(columns={"Rank": "Rank_P2"})

chart6_df = policy_full.join(t1).join(t2)

SECTOR_ORDER = ["Electricity", "Heating", "Transport"]
TECH_ORDER = {
    "Electricity": ["General", "Hydro Power", "PV & Wind", "CHP & CCGT"],
    "Heating": ["General", "HP", "Boiler"],
    "Transport": ["General", "FCEV", "EV"],
}
chart6_df["sector_rank"] = chart6_df["Energy_sector"].map({s: i for i, s in enumerate(SECTOR_ORDER)})
chart6_df["tech_rank"] = chart6_df.apply(
    lambda r: TECH_ORDER[r["Energy_sector"]].index(r["Promoting_technologies"]), axis=1)
chart6_df["status_rank"] = chart6_df["Status"].map({"I": 0, "N": 1})
chart6_df = chart6_df.sort_values(["sector_rank", "tech_rank", "status_rank"]).reset_index()

n_rows = len(chart6_df)
SECTOR_BAR_COLORS = {"Electricity": "#3E7CB1", "Heating": "#D2691E", "Transport": "#9B59B6"}
STATUS_TEXT = {"I": "Existing", "N": "Fictitious"}
P1_COLOR = "#2C3E50"
P2_COLOR = "#E8A33D"
IMPROVE_COLOR = "#2E7D32"
WORSEN_COLOR = "#C0392B"
P1_CAPSULE_COLOR = "#5B84A3"  # distinct steel-blue tint, ties to P1_COLOR's navy family
P2_CAPSULE_COLOR = "#E8A33D"  # unchanged grey, for Period 2

## Monte Carlo: most-frequent zone + % share, per policy per scenario
mc_topsis_p1 = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/MC_TOPSIS_2030_2040.csv")[["Iteration", "Policy_ID", "TOPSIS Score"]]
mc_topsis_p2 = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/MC_TOPSIS_2040_2050.csv")[["Iteration", "Policy_ID", "Rank"]]
mc_relevance = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/MC_Relevance.csv")

SCENARIOS = ["MIX", "REMIX", "H2"]


def classify_zone(group, topsis_score):
    if group in ("Strong negative", "Weak negative"):
        return group
    suffix = "_high" if topsis_score >= TOPSIS_SPLIT else "_low"
    return f"{group}{suffix}"

mode_zone = {}
for scen in SCENARIOS:
    mc_scen = mc_relevance[mc_relevance["Scenario"] == scen][["Iteration", "Policy_ID", "Group"]]
    merged = mc_scen.merge(mc_topsis_p1, on=["Iteration", "Policy_ID"])
    merged["Zone"] = [classify_zone(g, t) for g, t in zip(merged["Group"], merged["TOPSIS Score"])]
    zone_shares = merged.groupby("Policy_ID")["Zone"].value_counts(normalize=True)
    for pid in chart6_df["Policy_ID"]:
        top_zone = zone_shares[pid].idxmax()
        top_pct = zone_shares[pid].max()
        mode_zone[(pid, scen)] = (top_zone, top_pct)

## Monte Carlo: Period-1 and Period-2 rank percentiles per policy
mc_topsis_p1_ranks = pd.read_csv(r"D:/Tansy/Master thesis/MCDA/data/final/output/MC_TOPSIS_2030_2040.csv")[["Iteration", "Policy_ID", "Rank"]]
p1_rank_stats = mc_topsis_p1_ranks.groupby("Policy_ID")["Rank"].quantile([0.05, 0.25, 0.75, 0.95]).unstack()
p1_rank_stats.columns = ["P5", "P25", "P75", "P95"]

p2_rank_stats = mc_topsis_p2.groupby("Policy_ID")["Rank"].quantile([0.05, 0.25, 0.75, 0.95]).unstack()
p2_rank_stats.columns = ["P5", "P25", "P75", "P95"]

print("\nSample mode-zone results:")
for pid in chart6_df["Policy_ID"].head(3):
    for scen in SCENARIOS:
        z, p = mode_zone[(pid, scen)]
        print(f"  {pid} / {scen}: {z} ({p:.0%})")
print("\nSample P2 rank percentile spread:")
print(p2_rank_stats.head())

## Wrap names, compute a UNIFORM row height (same as Chart 2)
WRAP_WIDTH = 80
LINE_H = 0.155
INTER_GAP = 0.06

chart6_df["lines"] = chart6_df["Name"].apply(lambda n: textwrap.wrap(n, WRAP_WIDTH) or [n])
chart6_df["n_lines"] = chart6_df["lines"].apply(len)
max_lines = chart6_df["n_lines"].max()

ROW_H = max_lines * LINE_H
chart6_df["y_top"] = chart6_df.index * (ROW_H + INTER_GAP)
chart6_df["y_bottom"] = chart6_df["y_top"] + ROW_H
chart6_df["y_center"] = (chart6_df["y_top"] + chart6_df["y_bottom"]) / 2
table_height_in = chart6_df["y_bottom"].iloc[-1]

## Build the figure (same proportions/fonts as Chart 2)

fig_height = table_height_in + 2.6
fig = plt.figure(figsize=(16, fig_height))

width_ratios = [0.14, 3.3, 0.5, 1.6, 2.8]
top_margin_in = 1.5
bottom_margin_in = 1.3
gs = fig.add_gridspec(1, 5, width_ratios=width_ratios,
                       left=0.03, right=0.99,
                       bottom=bottom_margin_in / fig_height,
                       top=1 - top_margin_in / fig_height,
                       wspace=0.0)

ax_sector = fig.add_subplot(gs[0, 0])
ax_name = fig.add_subplot(gs[0, 1], sharey=ax_sector)
ax_status = fig.add_subplot(gs[0, 2], sharey=ax_sector)
ax_cat = fig.add_subplot(gs[0, 3], sharey=ax_sector)
ax_dot = fig.add_subplot(gs[0, 4], sharey=ax_sector)

for ax in [ax_sector, ax_name, ax_status, ax_cat, ax_dot]:
    ax.set_ylim(table_height_in, -0.35)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

### Sector indicator bar
ax_sector.set_xlim(0, 1)
for _, r in chart6_df.iterrows():
    ax_sector.add_patch(mpatches.Rectangle((0, r["y_top"]), 1, r["y_bottom"] - r["y_top"],
                         facecolor=SECTOR_BAR_COLORS[r["Energy_sector"]], edgecolor="none"))

### Policy name column
ax_name.set_xlim(0, 1)
for _, r in chart6_df.iterrows():
    n = len(r["lines"])
    start_y = r["y_center"] - (n - 1) * LINE_H / 2
    for i, line in enumerate(r["lines"]):
        ax_name.text(0.008, start_y + i * LINE_H, line, va="center", ha="left", fontsize=10)

### Status column
ax_status.set_xlim(0, 1)
for _, r in chart6_df.iterrows():
    ax_status.text(0.03, r["y_center"], STATUS_TEXT[r["Status"]], va="center", ha="left", fontsize=10)

### Category columns: IN-CELL BAR (mode zone % share) instead of solid fill
ax_cat.set_xlim(0, 3)
BAR_PAD, BAR_WIDTH = 0.03, 0.94
for _, r in chart6_df.iterrows():
    for i, scen in enumerate(SCENARIOS):
        zone, pct = mode_zone[(r["Policy_ID"], scen)]
        hex_color, _, textcolor = ZONE_COLORS[zone]
        cell_left = i + BAR_PAD
        ax_cat.add_patch(mpatches.Rectangle((cell_left, r["y_top"]), BAR_WIDTH, r["y_bottom"] - r["y_top"],
                          facecolor="#F5F5F5", edgecolor="white", linewidth=0.5, zorder=1))
        ax_cat.add_patch(mpatches.Rectangle((cell_left, r["y_top"]), BAR_WIDTH * pct, r["y_bottom"] - r["y_top"],
                          facecolor=hex_color, edgecolor="none", zorder=2))
        label_color = textcolor if pct > 0.22 else "#333333"
        ax_cat.text(cell_left + 0.03, r["y_center"], f"{pct:.0%}", ha="left", va="center",
                    fontsize=7.8, fontweight="bold", color=label_color, zorder=3)
ax_cat.set_xticks([0.5, 1.5, 2.5])
ax_cat.xaxis.set_ticks_position("top")
ax_cat.set_xticklabels(["MIX", "REMIX", "H2"], fontsize=10, fontweight="bold")
ax_cat.tick_params(axis="x", length=0, pad=-18)

### Rank change dot plot, WITH the added MC percentile capsule
changes = (chart6_df["Rank_P2"] - chart6_df["Rank_P1"])
det_max_abs_change = changes.abs().max()

p1_extremes = chart6_df.apply(
    lambda r: max(abs(p1_rank_stats.loc[r["Policy_ID"], "P95"] - r["Rank_P1"]),
                   abs(p1_rank_stats.loc[r["Policy_ID"], "P5"] - r["Rank_P1"])), axis=1)
p2_extremes = chart6_df.apply(
    lambda r: max(abs(p2_rank_stats.loc[r["Policy_ID"], "P95"] - r["Rank_P2"]),
                   abs(p2_rank_stats.loc[r["Policy_ID"], "P5"] - r["Rank_P2"])), axis=1)
max_abs_change = max(det_max_abs_change, p1_extremes.max(), p2_extremes.max())

MAX_OFFSET = 5
scale = MAX_OFFSET / max_abs_change if max_abs_change > 0 else 1
SPINE_X = 0
ax_dot.set_xlim(-MAX_OFFSET - 1.7, MAX_OFFSET + 1.5 )


def dx(rank, rank_p1):
    return - (rank - rank_p1) * scale

CAPSULE_HALF_H = ROW_H * 0.36

for _, r in chart6_df.iterrows():
    y = r["y_center"]
    pid = r["Policy_ID"]
    rank_p1 = r["Rank_P1"]
    change = int(r["Rank_P2"] - rank_p1)

    # Period 1 shade (upper band, steel-blue): 5th-95th percentile range

    y1p5, y1p95 = p1_rank_stats.loc[pid, ["P5", "P95"]]
    x1_a, x1_b = dx(y1p5, rank_p1), dx(y1p95, rank_p1)
    x1_left, cap1_w = min(x1_a, x1_b), max(abs(x1_b - x1_a), 0.05)
    ax_dot.add_patch(FancyBboxPatch(
        (x1_left, y - CAPSULE_HALF_H), cap1_w, 2 * CAPSULE_HALF_H,
        boxstyle=f"round,pad=0,rounding_size={CAPSULE_HALF_H}",
        facecolor=P1_CAPSULE_COLOR, edgecolor="none", alpha=0.45, zorder=1))



    # Period 2 shade (lower band, grey): 5th-95th percentile range
    p5, p95 = p2_rank_stats.loc[pid, ["P5", "P95"]]
    x2_a, x2_b = dx(p5, rank_p1), dx(p95, rank_p1)
    x2_left, cap2_w = min(x2_a, x2_b), max(abs(x2_b - x2_a), 0.05)
    ax_dot.add_patch(FancyBboxPatch(
        (x2_left, y - CAPSULE_HALF_H), cap2_w, 2 * CAPSULE_HALF_H,
        boxstyle=f"round,pad=0,rounding_size={CAPSULE_HALF_H}",
        facecolor=P2_CAPSULE_COLOR, edgecolor="none", alpha=0.4, zorder=1))

    # P1 dot + rank number (unchanged from Chart 2)
    ax_dot.scatter([SPINE_X], [y], s=250, c=P1_COLOR, edgecolors="white", linewidths=1.2, zorder=3)
    ax_dot.text(SPINE_X, y, str(int(rank_p1)), color="white", fontsize=9,
                ha="center", va="center", fontweight="bold", zorder=4)

    if change != 0:
        dx_p2 = dx(r["Rank_P2"], rank_p1)
        line_color = IMPROVE_COLOR if change < 0 else WORSEN_COLOR
        ax_dot.plot([SPINE_X, dx_p2], [y, y], color=line_color, linewidth=1.6, zorder=2, solid_capstyle="round")
        ax_dot.scatter([dx_p2], [y], s=250, c=P2_COLOR, edgecolors="white", linewidths=1.2, zorder=2.5)
        label = f"{change:+d}"
        label_x = dx_p2 + (0.35 if dx_p2 > 0 else -0.35)
        ax_dot.text(label_x, y, label, color=line_color, fontsize=9, fontweight="bold",
                    ha="left" if dx_p2 > 0 else "right", va="center", zorder=4)

## Column headers (identical to Chart 2)
fig.canvas.draw()
pos_name = ax_name.get_position()
pos_status = ax_status.get_position()
pos_cat = ax_cat.get_position()
pos_dot = ax_dot.get_position()

header_y = pos_name.y1 + 0.002
fig.text(pos_name.x0 + 0.18, header_y, "Policy", fontsize=12, fontweight="bold", va="bottom", ha="center")
fig.text(pos_status.x0 + 0.018, header_y, "Status", fontsize=12, fontweight="bold", va="bottom", ha="center")
fig.text((pos_cat.x0 + pos_cat.x1) / 2, header_y, "Scenario", fontsize=12, fontweight="bold", va="bottom", ha="center")
fig.text((pos_dot.x0 + pos_dot.x1) / 2, header_y, "TOPSIS Rank: 2030-2040 → 2040-2050",
          fontsize=12, fontweight="bold", va="bottom", ha="center")

## Title
fig.text(0.5, 0.995, "Policy Category Confidence & TOPSIS Rank Shift — Monte Carlo Uncertainty (10000 iterations)",
          fontsize=15, fontweight="bold", ha="center", va="top")

## Top legend: sector colors + dot colors + capsule note
sector_handles = [Line2D([0], [0], marker="s", color="none", markerfacecolor=c, markeredgecolor="none",
                          markersize=11, label=s) for s, c in SECTOR_BAR_COLORS.items()]
dot_handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor=P1_COLOR, markeredgecolor="white",
           markersize=11, label="2030-2040 TOPSIS Rank"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor=P2_COLOR, markeredgecolor="white",
           markersize=11, label="2040-2050 TOPSIS Rank"),
]
capsule_handles = [
    mpatches.Patch(facecolor=P1_CAPSULE_COLOR, alpha=0.45, label="MC 5th-95th percentile Period-1 rank"),
    mpatches.Patch(facecolor=P2_CAPSULE_COLOR, alpha=0.4, label="MC 5th-95th percentile Period-2 rank"),
]

legend_y = 0.975
leg1 = Legend(fig, sector_handles, [h.get_label() for h in sector_handles],
              loc="upper left", bbox_to_anchor=(0.03, legend_y - 0.007), bbox_transform=fig.transFigure,
              ncol=3, fontsize=10, frameon=False)
fig.add_artist(leg1)
dot_legend_x = (pos_dot.x0 + pos_dot.x1) / 2
leg2 = Legend(fig, dot_handles, [h.get_label() for h in dot_handles],
              loc="upper center", bbox_to_anchor=(dot_legend_x, legend_y), bbox_transform=fig.transFigure,
              ncol=2, fontsize=10, frameon=False)
fig.add_artist(leg2)
leg3 = Legend(fig, capsule_handles, [h.get_label() for h in capsule_handles],
              loc="upper center", bbox_to_anchor=(dot_legend_x, legend_y - 0.014), bbox_transform=fig.transFigure,
              ncol=2, fontsize=10, frameon=False)
fig.add_artist(leg3)

## Bottom legend: 8 zone colors (same style as Chart 1/2)
add_zone_legend(fig, x0=0.03, y0=bottom_margin_in / fig_height - 0.35 / fig_height,
                 width=0.94, is_fig=True, fontsize=10, row_gap=0.012)
fig.text(0.03, bottom_margin_in / fig_height - 0.95 / fig_height,
          "Category bars show each policy's most frequent zone across 10000 MC iterations; "
          "% = that zone's share of iterations.",
          fontsize=10, style="italic", ha="left", va="top")

fname = "Chart5_MC_uncertainty_table.png"
fig.savefig(fr"D:/Tansy/Master thesis/MCDA/data/final/output/{fname}", dpi=150, bbox_inches="tight", pad_inches=0.3)
plt.close(fig)
print(f"\n Saved: {fname}")


Sample mode-zone results:
  I_06 / MIX: Neutral_high (100%)
  I_06 / REMIX: Neutral_high (100%)
  I_06 / H2: Neutral_high (100%)
  I_15 / MIX: Neutral_high (100%)
  I_15 / REMIX: Neutral_high (100%)
  I_15 / H2: Neutral_high (100%)
  I_25 / MIX: Neutral_high (97%)
  I_25 / REMIX: Neutral_high (97%)
  I_25 / H2: Neutral_high (97%)

Sample P2 rank percentile spread:
             P5   P25   P75   P95
Policy_ID                        
I_01       37.0  38.0  41.0  42.0
I_03       25.0  26.0  31.0  34.0
I_04       37.0  38.0  41.0  42.0
I_05        8.0  11.0  18.0  22.0
I_06       14.0  18.0  24.0  24.0

 Saved: Chart5_MC_uncertainty_table.png
